In [ ]:
#| default_exp magic/web

In [ ]:
#| export
from __future__ import annotations

from datetime import datetime
import numpy as np
from dataclasses import dataclass, field
from typing import NamedTuple
from enum import Enum
from collections import defaultdict
import pandas as pd
import random
import math
import uuid
import httpx
import json
import logging
from itertools import combinations
from collections import deque


from monsterui.all import *
from fastcore.basics import patch

In [ ]:
import io, base64, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

In [ ]:
#| export
from fasthtml.common import *
from fasthtml.jupyter import *
from fasthtml.common import *

In [ ]:
#| export
from HexMagic.geology import Geology, DrainageBasins, Watershed
from HexMagic.overlay import  TerrainDisplay, TerrainOverlay, ClimateOverlay, TerraDemo, DrainageBasins, OverlaySpec
from HexMagic.game.flag import CountryFlag , PieceType, GameContext, DiagramGlyphs
from HexMagic.styles import StyleCSS,  SVGBuilder, SVGDef
from HexMagic.overlay import OverlaySpec, FlowOverlay, CreamOverlay, RiverOverlay, OceanWaveOverlay,Terrain
from HexMagic.overlay import ClimateOverlay, WatershedOverlay, TemperatureOverlay, PrecipitationOverlay,SoilOverlay,OverlaySpec
from HexMagic.climate import  TerraDemo, TerrainFactory
from HexMagic.primitives import HexTouchMap, HexPosition, HexGrid, HexDragMap, HexTouchMap, MapCord , PrimitiveDemo, Hex, HexWrapper
from HexMagic.primitives import MapRect, MapPath,MapCord,  MapSize, HexRegion
from HexMagic.water.soil import SoilSystem
from HexMagic.terrainpatterns import TerrainPatterns

In [ ]:
from HexMagic.magic.pieces import GameParts, Piece, Squad
from HexMagic.magic.pieces import PieceOverlay, LanternOverlay, SquadNamesOverlay
from HexMagic.magic.moves import Move, Rotation, Placement, Purchase, MoveOverlay, MoveLog
from HexMagic.magic.moves import  PendingMove

from HexMagic.magic.strategy import PlacementStrategy, BoardNeeds, OptimalPlacementOverlay

In [ ]:
from HexMagic.magic.pieces import GameParts, Piece, Squad, PieceType, TargetType, SquadTarget, PlacementStrategy, PlanContext,FogOverlay
from HexMagic.magic.food import FOOD_TABLE, FoodStats,FoodYield,FoodOverlay
 # Build main map
from HexMagic.magic.pieces import PieceOverlay, LanternOverlay, SquadNamesOverlay, HexNumberOverlay
from HexMagic.magic.moves import MoveError, Move, MoveLog, PurchasePreview,  Placement, Purchase, Rotation, PURCHASE_COST
from HexMagic.magic.network import SquadFlow, Squad,FoodNetworkOverlay,SquadFlowRenderer, SquadFlowDiagram
from HexMagic.magic.turns import GameRecord, TurnProcessor, TurnCell, TurnCharts


In [ ]:
SECRECT_KEY = "I don't know how to code"

# HexLib – AI Reference for FastHTML Refactoring

> Hex-based strategy game engine. Modules are autogenerated from nbdev notebooks
> in `../../nbs/magic/`. Goal: refactor `prerefactor.ipynb` into a FastHTML web app.

## Module Dependency Graph

```
food.py          ← standalone food/yield/overlay logic
board.py         ← imports food; defines GameParts (central state holder)
pieces.py        ← imports board, food; Piece/Squad/overlays
network.py       ← imports board, food, pieces; SquadFlow (networkx graphs)
moves.py         ← imports board, food, pieces; Move validation + UI panels
turns.py         ← imports all above; TurnProcessor game loop
strategy.py      ← imports all above; AI placement + Optuna tuning
```



## Core Classes

### GameParts (board.py) — Central State Object
Holds the world, grid, food yields, and squad list. Everything flows through this.

```python
gp = GameParts()                    # creates default world
gp.grid: HexGrid                    # hex grid with .hexes, .middle, .builder
gp.terr: Terrain                    # terrain with .elevations, .fields, .hexGrid
gp.fy: FoodYield                    # food yield calculator
gp.yields: np.ndarray               # computed yields
gp.basins: DrainageBasins           # drainage data
gp.squads: list[Squad]              # all squads in the game
gp.referenceIndex: int              # origin hex index

# Coordinate conversion (CRITICAL — used everywhere)
gp.hp2i(hexpos) -> int              # HexPosition → grid index
gp.i2hp(index) -> HexPosition       # grid index → HexPosition

# Location helpers
gp.location(hp, facing) -> MapPlace           # build MapPlace from HexPosition
gp.location_by_index(idx, facing) -> MapPlace # build MapPlace from index

# Overlay context for rendering
gp.overlayContext(region=None, padding=1, radius=None, ...) -> GameContext

# Game setup
gp.setup_game(n_players=5, year=1950, seed=23, starting_pantry=100) -> TurnProcessor
gp.loadSquads(year, seed, levels)
gp.find_starts(n, ring_radius=3, min_distance=6) -> list[int]
```

### MapPlace (board.py) — Position + State
```python
@dataclass
class MapPlace:
    location: HexPosition       # cube coordinates
    facing: HexPosition         # one of HexPosition.directions()
    level: int = 0              # elevation level at this hex
    harvest: int = 0            # food tier at this hex
    parent: GameParts | None    # back-reference to game state
    tenure: int = 0             # turns spent here
    pantry: int = 0             # per-piece food storage
```

### Piece (pieces.py) — Individual Game Piece
```python
@dataclass
class Piece:
    id: str                     # unique ID (name + uuid)
    flag: CountryFlag | None    # country/color identity
    type: PieceType             # KING, PAWN, BISHOP, KNIGHT, QUEEN
    place: MapPlace | None      # None if not on board
    name: str                   # character name
    year: int                   # birth year
    lantern: StyleCSS           # sight region fill style

# Key methods
piece.food -> FoodStats          # lookup from FOOD_TABLE
piece.sight() -> HexRegion       # visible hex indices
piece.icon(size=24) -> FT        # small inline SVG
piece.fetch_avatar() -> str      # DiceBear SVG (HTTP call)
piece.__ft__() -> FT             # full card rendering
```

### Squad (pieces.py) — Team of Pieces
```python
@dataclass
class Squad:
    pieces: list[Piece]
    name: str
    flag: CountryFlag | None
    strategy: PlacementStrategy
    target: SquadTarget
    queue: list[QueueEntry]      # purchase priority queue

# Key methods
squad.king_alive() -> bool
squad.to_nx() -> nx.DiGraph      # sight-based directed graph
squad.supply_bottlenecks() -> list[Piece]
squad.isolated_pieces() -> list[Piece]
squad.resolve_flow()             # centralized treasury
squad.orient_facings(parts)      # heuristic facing optimization
squad.optimize_facings(parts)    # coordinate descent optimization
squad.table() -> FT              # data table rendering
squad.__ft__() -> FT             # carousel card rendering

# Strategy (from strategy.py)
squad.reconsider(parts) -> bool  # re-evaluate strategy/target
squad.replan(parts)              # rebuild purchase queue
squad.execute_queue(parts) -> Move | None
squad.best_pawn_move(parts) -> Move | None
squad.best_bishop_move(parts) -> Move | None
squad.best_knight_move(parts) -> Move | None
squad.best_queen_move(parts) -> Move | None
```

### FoodStats (food.py) — Per-Type Economics
```python
class FoodStats(NamedTuple):
    diet: int           # food consumed per turn
    cost: int           # purchase cost
    harvest: float      # food production multiplier
    drain: float        # damage to enemies in sight
    bonus_sight: int    # extra sight range
    lifespan: int       # turns before piece expires

FOOD_TABLE = {
    PieceType.PAWN:   FoodStats(1,  3,  1.25, 1.0, 0, 36),
    PieceType.BISHOP: FoodStats(3,  9,  1.0,  0.75, 3, 5),
    PieceType.KNIGHT: FoodStats(2, 10,  0.0,  3.0, 0, 5),
    PieceType.QUEEN:  FoodStats(4, 18,  0.0,  4.5, 2, 20),
    PieceType.KING:   FoodStats(2,  0,  1.0,  0.5, 1, 999),
}
```

### Move (moves.py) — Turn Action
```python
@dataclass
class Move:
    rotations: list[Rotation]       # (piece, new_facing) pairs
    placement: Placement | None     # (piece, MapPlace)
    purchase: Purchase | None       # (PieceType, MapPlace, facing)

# Validation — raises MoveError on illegal moves
Move.from_squad(squad, rotations=[], placement=None, purchase=None) -> Move
move.apply(parts, squad)           # mutate board state
```

### TurnProcessor (turns.py) — Game Loop
```python
tp = TurnProcessor(parts, squads)
tp.resolve_move(squad, move) -> dict   # apply one squad's move, return summary
tp.play_round(moves: dict) -> list     # all squads move, advance turn
tp.status() -> pd.DataFrame            # current game state table
tp.record: GameRecord                  # historical snapshots
tp.log: MoveLog                        # move history
```

### SquadFlow (network.py) — Food Network Analysis
```python
flow = SquadFlow.from_squad(squad)     # full max-flow analysis
flow = SquadFlow.topology_only(squad)  # fast, no flow computation
flow.king_component -> set             # nodes connected to king
flow.bottlenecks -> set                # articulation points
flow.total_surplus -> int              # harvest - diet
flow.sight_projection(parts) -> float  # how well cones face target
```

## Overlay System (SVG Rendering)

Overlays are composable rendering layers. Each returns an `OverlaySpec`:

```python
OverlaySpec(name, render_fn, requires=set(), priority=int)
```

Higher priority renders on top. Use `TerrainDisplay(*overlays, ctx=ctx)` to
compose them. The render function receives a `GameContext` and returns SVG string.

### Common Overlays & Priority Order
```
CreamOverlay(stylized=True)     # base terrain colors      (low priority)
RiverOverlay(max_width=4)       # river lines
FoodOverlay(color="#558B2F")     # food density dots         priority=44
OasisOverlay(n=3)               # oasis highlights          priority=45
FogOverlay(fill="#ffffff")      # fog-of-war white          priority=6
LanternOverlay()                # piece sight cones         priority=62
PieceOverlay()                  # chess piece SVGs          priority=80
MoveOverlay(moves, squad)       # ghost move previews       priority=85
FoodNetworkOverlay()            # food flow edges/nodes     priority=85
SquadNamesOverlay()             # squad name labels         priority=90
HexNumberOverlay()              # hex index labels          priority=95
```

### Rendering a Map Snippet
```python
ctx = parts.overlayContext(region=hex_region, radius=20)
overlays = (CreamOverlay(stylized=True), RiverOverlay(max_width=4),
            LanternOverlay(), PieceOverlay())
TerrainDisplay(*overlays, ctx=ctx)
svg_xml = ctx.builder.xml()       # complete SVG string
ft_elem = Div(NotStr(svg_xml))    # wrap for FastHTML
```

## FastHTML Component Patterns

### Pattern: Return FT (FastTag) from `__ft__`
Every renderable class implements `__ft__(self) -> FT`. FastHTML auto-calls this
in notebook display and route handlers.

```python
# GOOD: composable FT tree
def __ft__(self):
    return Card(
        Table(Thead(...), Tbody(...)),
        header=(H4(self.name), P("subtitle", cls=TextPresets.muted_sm)),
    )
```

### Pattern: Inline SVG via NotStr
Raw SVG/HTML must be wrapped in `NotStr()` to prevent escaping:
```python
Div(NotStr(builder.xml()), style="line-height:0;")
Div(NotStr(flag.logo(size=24, piece_id="unique_id")))
```

### Pattern: MonsterUI Components
The codebase uses MonsterUI extensively for layout:
```python
DivFullySpaced(left, right)       # flex row, space-between
DivLAligned(icon, label, cls="gap-2")  # flex row, left-aligned
DivVStacked(*items, cls="gap-1")  # flex column
DivHStacked(*items)               # flex row
DivCentered(content)              # centered container
Card(*sections, header=..., footer=...)
Divider()
```

### Pattern: HTMX Interactivity
Buttons use hx_* attributes for server round-trips:
```python
Button("Commit Move", cls=ButtonT.primary,
       hx_post="/move/commit",
       hx_target="#move-panel",
       hx_swap="outerHTML")
```

### Pattern: Inline Styles (prevalent throughout)
Most styling is inline `style=` strings, not CSS classes:
```python
Div(content, style="display:flex; gap:8px; padding:4px;")
Span(text, style=f"color:{flag.darkPrimary}; font-weight:700;")
```

### Pattern: Flag-Colored Badges
```python
Span(f"🧺 {pantry}", style=f"font-size:1.2rem;font-weight:700;")
Span(squad.name, style=f"color:{squad.flag.darkPrimary};")
Div(content, style=f"background:{flag.lightComp};border-radius:8px;")
```

### Pattern: matplotlib → inline SVG
```python
fig, ax = plt.subplots(...)
# ... plot ...
buf = io.StringIO()
fig.savefig(buf, format='svg', bbox_inches='tight')
plt.close(fig)
return Div(NotStr(buf.getvalue()))
```

## Anti-Patterns to Avoid

### 1. Duplicate Definitions
`LanternOverlay` is defined TWICE in pieces.py (lines 431 and 685).
`PurchasePanel` is defined TWICE in moves.py (lines 623 and 790).
`hexes_in_range` is defined in BOTH food.py and strategy.py.
`QueueEntry` is defined in BOTH pieces.py and strategy.py.
`execute_queue` is patched in BOTH strategy.py (lines 344 and 1201).
→ Always check for existing definitions before adding new ones.

### 2. Module-Level Side Effects
`strategy.py` line 1431: `board = GameParts()` creates a world on import.
`strategy.py` line 1465: `study = optuna.create_study(...)` runs Optuna on import.
→ Guard with `if __name__ == "__main__"` or move to functions.

### 3. Mutable Back-References
`MapPlace.parent` points back to `GameParts`, creating circular refs.
`Piece.place.parent.hp2i(...)` is a 3-level chain used everywhere.
→ Be careful with serialization; these objects can't be pickled.

### 4. Duplicate Imports
Every file imports `from fasthtml.common import *` twice (lines like 15-17).
→ Remove the duplicate.

### 5. Comment-Commented Code
`pieces.py` has commented-out alternatives scattered throughout `__ft__`.
→ Clean up before refactoring.

### 6. Hardcoded Inline Styles
Extensive inline `style=` strings make theming difficult.
→ Prefer CSS classes or a style constants module for the web app.

### 7. PURCHASE_COST Referenced Before Import
`pieces.py` line 246 uses `PURCHASE_COST` but it's defined in `moves.py`.
This works at runtime because `@patch` defers execution, but is confusing.

## Key External Dependencies

| Package | Used For |
|---------|----------|
| fasthtml | Web framework, FT components, HTMX |
| monsterui | UI component library (Card, DivFullySpaced, etc.) |
| fastcore | `@patch` decorator for monkey-patching |
| networkx | Squad sight graphs, bottleneck detection |
| numpy | Terrain data, food yield computation |
| matplotlib | Chart generation (food yield __ft__) |
| pandas | MoveLog, GameRecord, turn summaries |
| pyomo + HiGHS | Optimal pawn placement MIP solver |
| optuna | Strategy hyperparameter tuning |
| httpx | DiceBear avatar fetching |

## HexPosition Coordinate System

Cube coordinates (q, r, s) where q + r + s = 0.
Six directions: `HexPosition.directions()` returns [E, NE, NW, W, SW, SE].
Field of view: `hp.field_of_view(facing, range)` returns visible HexPositions.
Distance: `hp.distance(other)` returns Chebyshev distance.

## CountryFlag API (from HexMagic.game.flag)

```python
flag = CountryFlag.seaborn("bright", levels=3)[0]
flag.primary       # primary color hex
flag.comp          # complementary color
flag.lightComp     # light complementary
flag.darkPrimary   # dark primary
flag.logo(size=24, piece_id="unique")           # SVG logo string
flag.pieceLogo(piece_type, size=48)             # piece icon SVG
flag.draw_piece(pt, center, builder, scale, size, piece_id)  # in-map piece
flag.piece_svg(pt, center, scale, size, piece_id)  # (svg_str, pat_def)
flag.make_squad(piece_types, year, seed) -> Squad
flag.labelStyle(name) -> StyleCSS
flag.pantry_badge(amount) -> FT
flag.net_badge(amount) -> FT
```

## SVGBuilder API (from HexMagic.styles)

```python
builder = SVGBuilder()
builder.width, builder.height = 600, 400
builder.add_style(StyleCSS(...))
builder.add_definition(SVGDef(...))
builder.add_font("Cinzel")
builder.adjust(layer_name, svg_string)   # append SVG to named layer
builder.xml() -> str                      # complete SVG document
```

## Web App Route Patterns (for new notebook)

The existing HTMX patterns suggest this route structure:
```python
@rt("/")           # main game view
@rt("/move/commit")           # POST: commit current move
@rt("/move/select_purchase")  # POST: select piece type to buy
@rt("/move/begin_rotation")   # POST: start rotating a piece
@rt("/move/clear")            # POST: clear pending move
```

Each route returns an FT fragment that replaces a target element via HTMX.

## Typical Game Setup Flow (for new notebook)

```python
parts = GameParts()                    # 1. create world
tp = parts.setup_game(n_players=3)     # 2. place kings at oases
# 3. Bootstrap pawns for each squad
for sq in parts.squads:
    sq.place_optimal_pawns(parts, tp, k=3)
# 4. Game loop
for turn in range(max_turns):
    moves = {}
    for sq in parts.squads:
        sq.reconsider(parts)           # AI re-evaluates strategy
        move = sq.execute_queue(parts) # AI picks best purchase
        if move: moves[sq] = move
    summaries = tp.play_round(moves)
    if tp.game_over: break
```

## Rendering Summary Cards (for web app panels)

```python
# Turn cell (compact summary per squad per turn)
tc = TurnCell.from_summary(summary_dict, squad)
tc.__ft__(squad)     # full card
tc.chips(squad)      # compact badge

# Turn table (grid: turns × squads)
tp.turnTable(summaries_by_turn)

# Move log
log.moveLogDisplay()
log.purchaseTable()

# Food network diagram
SquadFlowDiagram(squad, parts, width=600, height=400).__ft__()

# Full map with all overlays
ctx = parts.overlayContext()
TerrainDisplay(CreamOverlay(), RiverOverlay(), LanternOverlay(),
               PieceOverlay(), SquadNamesOverlay(), ctx=ctx)
Div(NotStr(ctx.builder.xml()))
```


# FastHTML Function Catalogue — HexLib

Every FastHTML, MonsterUI, and HTMX function used in this codebase, with
signatures, usage context, and concrete examples pulled from the code.

---

## 1. Core HTML Element Constructors

These come from `from fasthtml.common import *`. Each returns an `FT` (FastTag)
object. Children are positional args; HTML attributes are keyword args.

### `Div(*children, cls=None, id=None, style=None, **attrs) -> FT`

The workhorse container. Used hundreds of times. Accepts any mix of FT children,
strings, and `NotStr` for raw HTML.

```python
# Simple wrapper
Div(*sections, cls="space-y-2")

# Styled container
Div(content, style="display:flex; gap:8px; padding:4px;")

# With id for HTMX targeting
Div(panel_center(), id="center-panel")

# Nested structure (board.py:217-227)
Div(
    Div(NotStr(ctx.builder.xml()), style="line-height:0;"),
    Div(pill("🧺 3"), pill("⏳ 2t"),
        style="display:flex; justify-content:center;"),
    style="display:inline-block; border:1px solid #ddd; border-radius:6px;"
)
```

### `Span(text, cls=None, style=None, **attrs) -> FT`

Inline text element. Heavily used for badges, labels, styled text fragments.

```python
# Flag-colored text (moves.py:569)
Span(squad.name, style=f"font-size:11px;font-weight:700;color:{squad.flag.darkPrimary};")

# Emoji badge (turns.py:655)
Span(f"🧺{d_str}", style=f"font-size:11px;color:{color};font-weight:700;")

# Dot indicator (prerefactor.ipynb)
Span("●", style=f"color:{sq.flag.primary}; font-size:1.1em")
```

### `P(text, cls=None, **attrs) -> FT`

Paragraph. Often used with MonsterUI text presets.

```python
P("No moves recorded yet.", cls=TextPresets.muted_sm)
P(f"🐾 {self.animal} · {len(self.pieces)} pieces", cls=TextPresets.muted_sm)
P(f"⚠️ {result['error']}", cls="text-destructive font-medium text-center py-4")
```

### `H2(text, cls=None, **attrs) -> FT`
### `H3(text, cls=None, **attrs) -> FT`
### `H4(text, cls=None, **attrs) -> FT`

Headings. `H4` is the most common — used for card headers and section titles.

```python
H4("Move Log", cls="font-bold text-base")
H4(f"Turn {GS.turn}")
H3("⬡ HexMagic", cls="uk-h3")
H2("👑 Victory!", cls="text-yellow-500")
```

### `Small(text, cls=None, **attrs) -> FT`

Used for secondary info, timestamps, counts.

```python
Small(f"{log.turn_count()} turns · {len(log._moves)} moves", cls=TextPresets.muted_sm)
Small(pt.value.title(), cls="font-medium text-xs")
Small("±", cls="w-6 text-right text-muted-foreground text-xs")
```

### `Strong(text, cls=None, **attrs) -> FT`

Bold text, used for names and key values.

```python
Strong(pt.value.title(), cls="text-sm")
Strong(squad.name, cls="text-sm")
Strong("Connected")
```

### `Button(label, cls=None, disabled=False, **attrs) -> FT`

Interactive button. Almost always paired with HTMX attributes.

```python
# Ghost toggle (prerefactor.ipynb)
Button("▾", style="font-size:16px;background:none;border:none;cursor:pointer;",
       onclick=f"document.getElementById('{detail_id}').classList.toggle('hidden')")

# HTMX submit button (moves.py:745-748)
Button("Commit Move", cls=ButtonT.primary,
       hx_post="/move/commit",
       hx_target="#move-panel", hx_swap="outerHTML")

# Disabled state (prerefactor.ipynb)
Button("🛒 Buy", cls=(ButtonT.secondary, "text-sm flex-1"),
       hx_post="/purchase", hx_target="#piece-panel",
       disabled=pantry <= 8)
```

### `Label(text, cls=None, **attrs) -> FT`

MonsterUI label badge (not HTML `<label>`). Used for status indicators.

```python
Label(f"−{cost}🍞", cls=LabelT.destructive if affordable else LabelT.secondary)
Label(f"Hex {idx}", cls=LabelT.secondary)
Label("Reserve", cls=LabelT.destructive)
```

---

## 2. Table Components

### `Table(*children, style=None, **attrs) -> FT`
### `Thead(*rows) -> FT`
### `Tbody(*rows) -> FT`
### `Tr(*cells, style=None) -> FT`
### `Th(text, style=None, **attrs) -> FT`
### `Td(content, cls=None, style=None, **attrs) -> FT`

Standard HTML table elements. Used for squad rosters, move logs, turn history.

```python
# Squad roster table (pieces.py:330-352)
Card(
    Table(
        Thead(Tr(Th(""), Th("Name"), Th("Born"), Th("Loc"), Th("Facing"))),
        Tbody(*[Tr(
            Td(p.icon()),
            Td(p.name, cls=TextT.medium),
            Td(str(p.year), cls=TextPresets.muted_sm),
            Td(loc, cls=TextPresets.muted_sm),
            Td(facing, cls=TextPresets.muted_sm),
        ) for p in self.pieces]),
    ),
    header=(H4(self.name), P(f"🐾 {self.animal}", cls=TextPresets.muted_sm)),
)

# Turn grid with styled headers (turns.py:681-697)
Th(
    DivVStacked(
        NotStr(sq.flag.logo(size=28, piece_id=f"tth_{sq.name}")),
        Span(sq.name, style=f"font-size:10px;font-weight:700;color:{sq.flag.darkPrimary};"),
        style="align-items:center;gap:3px;"
    ),
    style=f"padding:8px;min-width:160px;text-align:center;"
          f"background:{sq.flag.lightComp};"
          f"border-bottom:2px solid {sq.flag.primary};"
)
```

---

## 3. MonsterUI Layout Components

From `from monsterui.all import *`. These generate flex/grid containers with
Tailwind-compatible classes.

### `DivFullySpaced(*children, cls=None) -> FT`

`display:flex; justify-content:space-between; align-items:center; width:100%`

The most-used layout helper. Places children at opposite ends of a row.

```python
# Header bar (moves.py:363-367)
DivFullySpaced(
    Div("Move", style="font-size:16px; font-weight:800;"),
    squad_badge,
    style="margin-bottom:12px;"
)

# Action row (prerefactor.ipynb)
DivFullySpaced(
    Button("🛒 Buy", cls=(ButtonT.secondary, "text-sm flex-1"),
           hx_post="/purchase", hx_target="#piece-panel"),
    Button("✅ Commit", cls=(ButtonT.primary, "text-sm flex-1"),
           hx_post="/commit", hx_target="body"),
    cls="gap-2",
)

# Stat row (prerefactor.ipynb)
DivFullySpaced(Small("Net"), Small(f"{net:+d}", cls="text-green-600 font-bold"))
```

### `DivLAligned(*children, cls=None) -> FT`

`display:flex; justify-content:flex-start; align-items:center`

Left-aligned row. Used for icon + label pairs.

```python
# Icon + text (moves.py:816)
DivLAligned(piece_logo_html(flag, pt, size=56), label, cls="gap-3")

# Stat indicator (prerefactor.ipynb)
DivLAligned(Span("🔗", cls=TextT.lg), Strong("Connected"))

# Tab bar (prerefactor.ipynb)
DivLAligned(
    Button("🗺️ Map", cls=ButtonT.primary, hx_get="/panel/center?view=map", ...),
    Button("👥 Squad", cls=ButtonT.ghost, hx_get="/panel/center?view=squad", ...),
    cls="gap-1 pb-2",
)
```

### `DivVStacked(*children, cls=None) -> FT`

`display:flex; flex-direction:column`

Vertical stack. Used for card content, stat groups, label stacks.

```python
# Purchase row label (moves.py:809-813)
DivVStacked(
    Strong(pt.value.title(), cls="text-sm"),
    cost_badge,
    pantry_note,
)

# Button rows (moves.py:721-725)
DivVStacked(*rows, cls="gap-2") if rows else P("No pieces on board.", cls=TextPresets.muted_sm)
```

### `DivHStacked(*children, style=None) -> FT`

`display:flex; flex-direction:row`

Horizontal stack. Used for footer stat bars.

```python
# Network footer (network.py:420-430)
DivHStacked(
    flag.pantry_badge(king_pantry),
    flag.net_badge(flow.total_surplus),
    Span(f"🔄 {flow.total_flow:.0f}", style="font-size:1.1rem;font-weight:700;"),
    Span(f"⚠️ {bn_count} bottlenecks", style=f"font-size:0.85rem;color:{bn_color};"),
    style="gap:16px; align-items:center; flex-wrap:wrap; padding:4px 8px;"
)
```

### `DivCentered(*children) -> FT`

Centered container. Used for full-page layouts and end-game screens.

```python
# Game over screen (prerefactor.ipynb)
DivCentered(
    H2("☠️ Your Kingdom Falls", cls="text-destructive"),
    P(f"The treasury ran dry on turn {GS.turn}.", cls=TextPresets.muted_lg),
    _game_over_summary(),
    Button("🏠 New Game", cls=(ButtonT.primary, "mt-4"),
           hx_get="/", hx_target="body"),
)

# Squad card wrapper (pieces.py:385)
DivCentered(header, Card(*sections, cls='space-y-4'))
```

### `Grid(*children, cols_sm=None, cols_md=None, cols_lg=None, cls=None) -> FT`

Responsive CSS Grid. Used for card grids and stat panels.

```python
# Network health stats (prerefactor.ipynb)
Grid(
    Div(DivLAligned(Span("🔗", cls=TextT.lg), Strong("Connected")),
        P(f"{len(G.nodes)} pieces", cls=TextPresets.muted_sm)),
    Div(DivLAligned(Span("⚠️", cls=TextT.lg), Strong("Bottlenecks")),
        P(f"{len(bottlenecks)}", cls=TextPresets.muted_sm)),
    cols_sm=2, cols_md=4,
)

# Piece card grid (prerefactor.ipynb)
Grid(*cards, cols_sm=2, cols_md=3, cols_lg=4, cls="gap-3")
```

---

## 4. MonsterUI Compound Components

### `Card(*body, header=None, footer=None, cls=None, id=None) -> FT`

Bordered card with optional header/footer slots. The primary container for
panels, tables, and detail views.

```python
# Simple data card (prerefactor.ipynb)
Card(
    Table(Thead(...), Tbody(...)),
    header=H4(f"Turn {GS.turn}"),
)

# Full card with footer (network.py:454-461)
Card(
    NotStr(builder.xml()),
    footer=self._render_footer(flow),
    header=DivFullySpaced(
        Div(H4(self.squad.name), P(f"{flag.name} · {placed_count} placed")),
    ),
)

# Move panel (moves.py:735-752)
Card(
    PurchasePanel(flag, king_pantry, selected_purchase_type),
    Divider(),
    PlacementInfo(selected_purchase_type, target_hex),
    Divider(),
    RotationPanel(pending_rotations, squad),
    Divider(),
    DivFullySpaced(
        Button("Clear", cls=ButtonT.ghost, hx_post="/move/clear", ...),
        Button("Commit Move", cls=ButtonT.primary, hx_post="/move/commit", ...),
    ),
    id="move-panel", cls="p-4 space-y-3 w-72",
)

# Piece card with header + footer + stats (prerefactor.ipynb)
Card(
    DivLAligned(avatar, Div(DivLAligned(Span(p.type.icon), Strong(p.name)),
                            P(f"b. {p.year}", cls=TextPresets.muted_sm))),
    Divider(cls="my-2"),
    stats,
    header=DivFullySpaced(
        Small(p.type.value.title(), cls=TextPresets.muted_sm),
        location_badge,
    ),
    footer=DivFullySpaced(
        Small(f"👁️ {sight_count} hexes", cls=TextPresets.muted_sm),
        Button("🔍", cls=(ButtonT.ghost, "text-xs"),
               hx_post="/piece/focus", hx_vals=json.dumps({"piece_id": p.id}),
               hx_target="#center-panel"),
    ),
    cls=f"{'ring-1 ring-primary' if on_board else 'opacity-70'}",
)
```

### `Divider(cls=None) -> FT`

Horizontal rule / separator. Used between Card sections.

```python
Divider()
Divider(cls="my-2")
```

### `Container(*children) -> FT`

Page-level centered container (max-width wrapper).

```python
# Full page (prerefactor.ipynb)
Title("HexMagic — Game"), Container(
    DivFullySpaced(H3("⬡ HexMagic"), ...),
    Divider(),
    Div(..., cls="flex gap-3"),
)
```

### `Title(text) -> FT`

Sets the HTML `<title>`. Returned alongside page content.

```python
return Title("HexMagic — Game"), Container(...)
return Title("Eliminated"), Container(...)
```

---

## 5. Text & Button Presets (MonsterUI)

### `TextPresets`

CSS class bundles for common text styles:

- `TextPresets.muted_sm` — gray, small (the most frequently used preset)
- `TextPresets.muted_lg` — gray, larger

```python
P("No pieces on board yet.", cls=TextPresets.muted_sm)
P(f"The treasury ran dry.", cls=TextPresets.muted_lg)
```

### `TextT`

Individual text style tokens:

- `TextT.medium` — medium weight
- `TextT.lg` — large size

```python
Td(p.name, cls=TextT.medium)
Span("🔗", cls=TextT.lg)
Span(f"Turn {GS.turn}", cls=TextT.lg)
```

### `ButtonT`

Button variant classes:

- `ButtonT.primary` — filled primary color
- `ButtonT.secondary` — filled secondary
- `ButtonT.ghost` — transparent, text-only

```python
Button("Commit", cls=ButtonT.primary)
Button("Clear", cls=ButtonT.ghost)
Button("🛒 Buy", cls=(ButtonT.secondary, "text-sm flex-1"))  # tuple = multiple classes
```

### `LabelT`

Label badge variants:

- `LabelT.destructive` — red/danger
- `LabelT.secondary` — muted

```python
Label(f"−{cost}🍞", cls=LabelT.destructive)
Label("Reserve", cls=LabelT.destructive)
Label(f"Hex {idx}", cls=LabelT.secondary)
```

---

## 6. Raw HTML Injection

### `NotStr(html_string) -> FT`

Wraps a raw HTML/SVG string so FastHTML does **not** escape it. Critical for
inline SVG rendering.

```python
# SVG map (board.py:218)
Div(NotStr(ctx.builder.xml()), style="line-height:0;")

# Flag logo (pieces.py:476)
Div(NotStr(self.flag.pieceLogo(piece_type=self.type, size=48)))

# Squad logo in table (network.py:458)
NotStr(squad.flag.logo(size=28, piece_id=f"tth_{sq.name}"))

# Matplotlib SVG chart (food.py:369)
Div(NotStr(buf.getvalue()), P("temp_curve · soil_mult", cls="text-xs opacity-50 text-center"))
```

---

## 7. HTMX Attributes

HTMX attributes are passed as Python keyword args with underscores replacing
hyphens. These drive all interactivity in the web app.

### `hx_get` / `hx_post`

Trigger HTTP requests on user interaction. `hx_post` is used for state mutations;
`hx_get` for reads.

```python
# GET — lazy-load panel on page load
Div(hx_get="/panel/pieces", hx_trigger="load",
    hx_target="#piece-panel", id="piece-panel")

# POST — commit a move
Button("✅ Commit", hx_post="/commit", hx_target="body")

# POST — select piece type to buy
Button("Select", hx_post="/purchase/select",
       hx_vals=json.dumps({"piece_type": pt.value}),
       hx_target="#piece-panel")
```

### `hx_target`

CSS selector for where to put the response HTML.

```python
hx_target="#piece-panel"       # replace contents of #piece-panel
hx_target="#center-panel"      # replace center panel
hx_target="body"               # replace entire page (for redirects)
hx_target="#move-panel"        # replace move panel
hx_target="#left-col"          # replace left sidebar
hx_target="#chart-panel"       # replace chart area
hx_target="#history-panel"     # replace history sidebar
```

### `hx_swap`

How the response replaces the target. Default is `innerHTML`.

```python
hx_swap="outerHTML"            # replace the entire target element (including its tag)
hx_swap="innerHTML"            # replace only the target's children (default)
```

### `hx_swap_oob`

Out-of-band swap — update a **different** element alongside the main response.
This is the key pattern for updating multiple panels from one request.

```python
# Return pieces panel as main response, PLUS update the map
map_update = Div(panel_center(), hx_swap_oob="innerHTML:#center-panel")
return panel_pieces(), map_update

# Return map as main, PLUS update pieces sidebar
pieces_update = Div(panel_pieces(), hx_swap_oob="innerHTML:#piece-panel")
return panel_center(), pieces_update
```

### `hx_vals`

JSON string of extra values to send with the request.

```python
hx_vals=f'{{"piece_type": "{pt.value}"}}'
hx_vals=json.dumps({"piece_id": p.id})
hx_vals=json.dumps({"col": key})
hx_vals=json.dumps({"squad": sq.name})
```

### `hx_trigger`

When to fire the request. Used for lazy-loading on page render.

```python
Div(hx_get="/panel/pieces", hx_trigger="load", hx_target="#piece-panel", id="piece-panel")
Div(hx_get="/panel/center", hx_trigger="load", hx_target="#center-panel", hx_swap="outerHTML")
Div(hx_get="/panel/history", hx_trigger="load", hx_target="#history-panel")
```

---

## 8. Routing

### `@rt(path, methods=None)` decorator

Registers a FastHTML route. From `from fasthtml.common import *` (actually from
the app created by `fast_app()`).

```python
# GET route (default)
@rt("/panel/center")
def panel_center(view: str = "map"):
    return Div(tabs, content, id="center-panel")

# POST route
@rt("/commit", methods=["POST"])
def commit():
    result = GS.commit_move()
    ...
    return RedirectResponse("/game", status_code=303)

# Bare decorator (auto-names route from function name)
@rt
def game():
    return Title("HexMagic — Game"), Container(...)
```

### Route parameters

Parameters are extracted from query strings (GET) or form data (POST):

```python
@rt("/hex_click", methods=["POST"])
def hex_click(hex_id: int):         # from form data

@rt("/panel/center")
def panel_center(view: str = "map"): # from ?view=map query param

@rt("/piece/select", methods=["POST"])
def piece_select(piece_id: str):     # from form data

@rt("/zoom", methods=["POST"])
def zoom(radius: int):               # from form data
```

### `RedirectResponse(url, status_code=303)`

From Starlette. Used after commits to do a full page refresh.

```python
return RedirectResponse("/game", status_code=303)
return RedirectResponse("/", status_code=303)
```

---

## 9. FastHTML Jupyter Integration

### `from fasthtml.jupyter import *`

Imported by every module. Provides notebook-specific display integration.

### `show(ft_element)`

Display an FT element in a Jupyter notebook as rendered HTML.

```python
show(diagram)
show(squad)    # calls squad.__ft__() automatically
```

### `__ft__(self) -> FT`

Magic method. When an object has `__ft__`, FastHTML auto-renders it in notebooks
and route responses. This is the primary rendering protocol.

```python
# Piece card (pieces.py:469)
@patch
def __ft__(self: Piece):
    ...
    return Div(inner, stats_row, style=f"border-radius:24px; ...")

# Squad carousel (pieces.py:357)
@patch
def __ft__(self: Squad):
    ...
    return DivCentered(header, Card(*sections, cls='space-y-4'))

# Food yield charts (food.py:293)
@patch
def __ft__(self: FoodYield):
    ...
    return Div(NotStr(buf.getvalue()), P(...))

# Turn cell (turns.py:581)
@patch
def __ft__(self: TurnCell, squad: Squad) -> FT:
    ...
    return Div(pantry_row, food_row, chips_row, *elim, style=...)

# Flow diagram (network.py:310)
def __ft__(self):
    return self._render()
```

---

## 10. Page Structure Pattern

The prerefactor.ipynb establishes a three-column layout loaded via HTMX:

```python
# Main game page
Title("HexMagic"), Container(
    # Top bar
    DivFullySpaced(H3("⬡ HexMagic"), turn_info, menu_button, cls="py-2"),
    Divider(),
    # Three-column flex layout
    Div(
        # Left: piece panel (lazy-loaded)
        Div(hx_get="/panel/pieces", hx_trigger="load",
            hx_target="#piece-panel", id="piece-panel",
            cls="w-[250px] flex-shrink-0 overflow-y-auto max-h-[80vh]"),

        # Center: map/squad/flow/charts (lazy-loaded, tab-switchable)
        Div(id="center-panel",
            hx_get="/panel/center", hx_trigger="load",
            hx_target="#center-panel", hx_swap="outerHTML",
            cls="flex-1 min-w-0"),

        # Right: history + debug (lazy-loaded)
        Div(
            Div(id="history-panel",
                hx_get="/panel/history", hx_trigger="load",
                hx_target="#history-panel"),
            cls="w-[280px] flex-shrink-0 overflow-y-auto max-h-[80vh]"),

        cls="flex gap-3",
    ),
)
```

Each panel is an independent route that returns an FT fragment. Panels update
each other via `hx_swap_oob` for coordinated UI updates.

---

## 11. Common Patterns Summary

| Pattern | Example | Where |
|---------|---------|-------|
| Card + Table | `Card(Table(Thead, Tbody), header=H4(...))` | pieces, turns, moves |
| Lazy-load panel | `Div(hx_get="/route", hx_trigger="load")` | game page |
| Multi-panel update | `return main_panel(), Div(other, hx_swap_oob="innerHTML:#id")` | hex_click, select |
| Tab switcher | `Button(cls=ButtonT.primary if active else ButtonT.ghost, hx_get=...)` | center panel |
| Stat badge | `DivFullySpaced(Small("Label"), Small(value, cls="font-bold"))` | piece cards |
| Inline SVG | `Div(NotStr(builder.xml()), style="line-height:0;")` | map, logos |
| matplotlib chart | `fig.savefig(buf, format='svg'); Div(NotStr(buf.getvalue()))` | food, charts |
| Disabled button | `Button(..., disabled=not affordable)` | purchase panel |
| Toggle state | `hx_post="/toggle_X"` → flip GS flag → re-render panel | overlays, charts |
| Collapsible | `onclick="el.classList.toggle('hidden')"` / cls="hidden" | move log |


I am going to remove the cating of the entire directory. Is there anything I should add that isn't in the notes that you would find helpful?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good timing to check! The notes are impressively thorough. A few gaps I notice that could help during refactoring:

1. **Global game state (`GS`)** — routes reference it everywhere (e.g. `GS.turn`, `GS.commit_move()`) but it's not documented. What is it? A singleton? A module-level object? Its full API would be very useful.

2. **`fast_app()` initialization** — the app setup pattern (how `rt` is obtained, middleware, static files, etc.) isn't shown.

3. **`GameContext`** — returned by `overlayContext()` and passed to all overlay render functions, but its fields/API aren't documented.

4. **`PlacementStrategy` / `SquadTarget` / `QueueEntry`** — referenced in `Squad` but their values/fields aren't listed.

5. **`MoveError`** — raised by `Move.from_squad()` but not documented (fields, how it's caught in routes).

6. **Known bugs / TODO list** — if there are any known issues beyond the anti-patterns already listed, those would save debugging time.

Which of those feel most relevant to your refactoring work? Or is there a specific area you're planning to tackle first?

Good timing to check! The notes are impressively thorough. A few gaps I notice that could help during refactoring:

1. **Global game state (`GS`)** — routes reference it everywhere (e.g. `GS.turn`, `GS.commit_move()`) but it's not documented. What is it? A singleton? A module-level object? Its full API would be very useful.

2. **`fast_app()` initialization** — the app setup pattern (how `rt` is obtained, middleware, static files, etc.) isn't shown.

3. **`GameContext`** — returned by `overlayContext()` and passed to all overlay render functions
```python
@dataclass
class GameContext(OverlayContext):
    """Rendering context with game-layer data (board, corridors, pieces)."""

    @property
    def board(self): return self.extras.get('board')

    @property
    def corridors(self): return self.extras.get('corridors', [])

    @property
    def pieces(self): return self.extras.get('pieces', [])

    @property
    def squads(self): return self.extras.get('squads', [])

    @property
    def simulator(self):
        return self.extras.get('simulator')
```

4. **`PlacementStrategy` / `SquadTarget` / `QueueEntry`** — referenced in `Squad` but their values/fields aren't listed.
```

```python
@dataclass
class PlacementStrategy:
    """Squad personality that shapes piece priority and stance thresholds."""
    name: str
    # Stance thresholds (how easily this personality shifts)
    defend_threat: int = 2       # enemy pieces near king to trigger defend
    attack_pantry: int = 60      # min king pantry to go offensive
    attack_placed: int = 4       # min pieces on board before attacking
    # Knight eagerness: lower = fires more readily
    knight_roi_mult: float = 1.0  # multiplied into knight ROI gate
    # NEW: spending discipline
    min_reserve_per_piece: int = 5    # pantry reserve per placed piece
    min_budget_ratio: float = 1.0     # won't buy unless income/upkeep >= this
    max_deficit_turns: int = 2        # skip buying if net negative for N+ turns
    buy_cooldown: int = 0             # min turns between purchases (0=every turn)
    afforable_min = 3                 # min number of pieces before we check on afford
    target_composition: dict[PieceType, int] = field(default_factory=dict)
    graduation: str = "balanced"

    @property
    def min_pawns(self: PlacementStrategy) -> int:
        """Pawn count from target_composition, with a sensible floor."""
        return max(2, self.target_composition.get(PieceType.PAWN, 2))
```

**File:** `~/HexMagic/HexMagic/magic/pieces.py`
```python
PlacementStrategy.AGGRESSIVE = PlacementStrategy(
    "aggressive",
    defend_threat=1, attack_pantry=40, attack_placed=3,
    knight_roi_mult=0.7, min_reserve_per_piece=3,
    min_budget_ratio=0.6, buy_cooldown=0,
    target_composition={
        PieceType.PAWN:   5,
        PieceType.KNIGHT: 3,
        PieceType.BISHOP: 3,
    },
    graduation="random")

PlacementStrategy.TURTLE = PlacementStrategy(
    "turtle",
    defend_threat=1, attack_pantry=999, attack_placed=99,
    min_reserve_per_piece=4, min_budget_ratio=1.0, buy_cooldown=0,
    target_composition={
        PieceType.PAWN:   6,
        PieceType.BISHOP: 5,
        PieceType.QUEEN:  2,
    },
    graduation="balanced")

PlacementStrategy.EXPANSIVE = PlacementStrategy(
    "expansive",
    defend_threat=3, attack_pantry=80, attack_placed=6,
    min_reserve_per_piece=4, min_budget_ratio=0.8, buy_cooldown=0,
    target_composition={
        PieceType.PAWN:   4,
        PieceType.BISHOP: 5,  # bishops = sight range = territory
    },
    graduation="turtle")   # once you reach oasis, sit on it

PlacementStrategy.BALANCED = PlacementStrategy(
    "balanced",
    defend_threat=2, attack_pantry=60, attack_placed=4,
    min_reserve_per_piece=5, min_budget_ratio=1.0, buy_cooldown=0,
    target_composition={
        PieceType.PAWN:   4,
        PieceType.BISHOP: 2,
        PieceType.KNIGHT: 1,
        PieceType.QUEEN:  1,
    },
    graduation="aggressive")
```

5. **`MoveError`** — raised by `Move.from_squad()` but not documented (fields, how it's caught in routes).

6. **Known bugs / TODO list** — if there are any known issues beyond the anti-patterns already listed, those would save debugging time.

Which of those feel most relevant to your refactoring work? Or is there a specific area you're planning to tackle first?

In [ ]:
??PendingMove

In [ ]:
#| export
@dataclass
class GlobalGameState:
    parts: GameParts | None = None
    tp: TurnProcessor | None = None
    player_squad: Squad | None = None
    pendingMove: PendingMove | None = None
    active_overlays: set = field(default_factory=lambda: {'rivers','lanterns','network','ocean'})
    turn: int = 0
    radius: float = 25
    debug_overlays: bool = False
    rotationClickId: int = None
    ai_strategies: dict[str, PlacementStrategy] = field(default_factory=dict)
    focused_piece: Piece | None = None
    charts: TurnCharts | None = None

    def select_piece(self, piece: Piece):
        """Select an unplaced piece for placement."""
        if piece.place is not None:
            return  # already on board — rotation handled separately
        self.pendingMove.selected_piece = piece

    def place_piece(self, hex_idx: int) -> bool:
        """Place the selected piece at a hex."""
        pm = self.pendingMove
        if pm is None or pm.selected_piece is None:
            return False
        loc = self.parts.location_by_index(hex_idx)
        pm.set_placement(pm.selected_piece, loc)
        pm.selected_piece.place = loc   # preview immediately
        return True

    def unplace_piece(self, piece: Piece):
        """Remove a pending placement (before committing)."""
        pm = self.pendingMove
        if pm and pm.placement and pm.placement.piece is piece:
            piece.place = None
            pm.placement = None

    @property
    def is_active(self) -> bool:
        return self.parts is not None and self.tp is not None

    @property
    def game_over(self) -> bool:
        return self.tp.game_over if self.tp else False
    
    # Convenience pass-throughs so routes don't reach into pendingMove directly
    @property
    def selected_piece(self) -> Piece | None:
        return self.pendingMove.selected_piece if self.pendingMove else None
    
    @property
    def has_pending(self) -> bool:
        return self.pendingMove is not None and self.pendingMove.has_actions


@patch
def start_game(self:GlobalGameState, size=300, player_count=3, seed=23, radius=20,debug=False):
    random.seed(seed)
    np.random.seed(seed)
    starting_pantry = 100

    world = TerrainFactory.create_world(
                bounds=MapRect(MapCord(0, 0), MapSize(size, size)),
                preset='temperate',
                name='Maiden Lane',
                radius=radius,
                lon_span=10.0,
                num_plates=8,
                subdivisions=3,
                ocean_fraction=0.1,
                oceanic_sides=['N'],
                terrain_age='young',
                formation_type='ridge',
                elevation_scale=1.5,
                erosion_age=0.1,
                num_lakes=0,
                seed=seed,
                debug=debug
            )

    

    self.parts = GameParts(world)
    debugParts = self.parts


    if debug:
        debugParts.terr.__ft__()
        debugParts.fy.summary()
        show(debugParts.terr)
    self.parts.grid.adjustRadius(radius)

    oasis_idxs = self.parts.find_starts(n=player_count, ring_radius=3, min_distance=9)
    self.parts.flags = CountryFlag.seaborn("bright",levels=player_count)
    year = 1880
    for i,flag in enumerate(self.parts.flags):
        roster = {
            #PieceType.PAWN: 3, 
            #PieceType.BISHOP: 2, 
            #PieceType.QUEEN: 2, 
            PieceType.KING: 1, 
            #PieceType.KNIGHT: 4
        }
        pieceKinds = [pt for pt, n in roster.items() for _ in range(n)]

        squad = flag.make_squad(pieceKinds, year=year) 
        king = next(p for p in squad.pieces if p.type == PieceType.KING)
        king.place = self.parts.location_by_index(oasis_idxs[i])  
        king.place.pantry = starting_pantry
        squad.place_optimal_pawns(self.parts, self.tp, k=3)
        self.parts.squads.append(squad)

    self.tp = TurnProcessor(self.parts, self.parts.squads)
    self.player_squad = self.parts.squads[0]
    
    ai_pool = [PlacementStrategy.AGGRESSIVE, PlacementStrategy.EXPANSIVE,
                PlacementStrategy.TURTLE, PlacementStrategy.BALANCED]

    self.ai_strategies = {sq.name: ai_pool[i % len(ai_pool)]
                          for i, sq in enumerate(self.parts.squads[1:])}
    self.selected_piece = None
    self.pendingMove = PendingMove()
    self.charts = TurnCharts(tp=self.tp, player_squad=self.player_squad)
    #self.active_chart_cols = {'king_pantry', 'pieces_placed'}
    #self.charts.active_squads = set()   # ← add these two lines
    self.turn = 0


debugPats = GameParts()
show(debugPats.terr)

In [ ]:
debugGS = GlobalGameState()
debugGS.start_game()

Simulation code

```python
def run_trial_bal_exp(parts: GameParts, 
                      bal_comp: dict[PieceType, int],
                      exp_comp: dict[PieceType, int],
                      bal_weights: tuple[float, ...],
                      exp_weights: tuple[float, ...],
                      starting_pantry: int = 100,
                      seed: int = 42,
                      verbose: bool = False) -> dict:
    """Run one Balanced vs Expansive game. Returns metrics dict."""
    
    if not hasattr(parts, '_oasis_idxs'):
        parts._oasis_idxs = parts.fy.find_oasis(3, min_distance=6)
    
    parts.squads.clear()
    parts.loadSquads(year=1950, seed=seed, levels=2)
    
    for squad in parts.squads:
        for p in squad.pieces:
            p.place = None
    
    starts = parts.find_starts(n=2, ring_radius=3, min_distance=9)
    for squad, start_idx in zip(parts.squads, starts):
        king = next(p for p in squad.pieces if p.type == PieceType.KING)
        loc = parts.location_by_index(start_idx)
        Move.from_squad(squad, placement=Placement(king, loc)).apply(parts)
        king.place.pantry = starting_pantry
    
    bal_squad, exp_squad = parts.squads[0], parts.squads[1]
    
    bal_strategy = PlacementStrategy(
        "balanced", defend_threat=2, attack_pantry=60, attack_placed=4,
        min_reserve_per_piece=5, min_budget_ratio=1.0,
        buy_cooldown=0, target_composition=bal_comp, graduation="aggressive")
    
    exp_strategy = PlacementStrategy(
        "expansive", defend_threat=3, attack_pantry=80, attack_placed=6,
        min_reserve_per_piece=4, min_budget_ratio=0.8,
        buy_cooldown=0, target_composition=exp_comp, graduation="aggressive")
    
    bal_squad.strategy = bal_strategy
    exp_squad.strategy = exp_strategy
    
    trial_weights = {
        "balanced": {"expand": bal_weights, "defend": bal_weights, "attack": bal_weights},
        "expansive": {"expand": exp_weights, "defend": exp_weights, "attack": exp_weights},
    }
    
    tp = TurnProcessor(parts, parts.squads)
    
    for squad in parts.squads:
        squad.place_optimal_pawns(parts, tp, k=3)
    
    total_pieces = sum(bal_comp.values()) + sum(exp_comp.values())
    max_rounds = total_pieces * 4 + 20
    
    # Accumulators
    bal_harvest_total = 0.0
    bal_drain_dealt = 0.0
    bal_drain_received = 0.0
    exp_sight_sum = 0.0  # sum of sight coverage each turn (for averaging)
    exp_sight_samples = 0
    errors = 0
    
    land_set = {i for i in range(len(parts.terr.elevations))
                if parts.terr.elevations[i] > 0}
    
    old_obj = dict(PURCHASE_OBJ)
    PURCHASE_OBJ.update(trial_weights)
    
    try:
        for turn in range(max_rounds):
            if tp.game_over:
                break
            
            for squad in parts.squads:
                if not squad.king_alive():
                    continue
                
                old_strat = squad.strategy
                squad.reconsider(parts)
                squad.strategy = old_strat
                
                move = squad.execute_queue(parts)
                if move is not None:
                    try:
                        summary = tp.resolve_move(squad, move)
                        if squad is bal_squad:
                            bal_harvest_total += summary.get('food_harvested', 0)
                            bal_drain_dealt += summary.get('food_drain_dealt', 0)
                            bal_drain_received += summary.get('food_drain_received', 0)
                    except MoveError:
                        errors += 1
                        squad.queue.clear()
                        squad.plan_ctx = None
            
            if tp.game_over:
                break
            
            # Measure expansive sight before food round
            exp_placed = [p for p in exp_squad.pieces if p.place is not None]
            if exp_placed:
                visible = set()
                for p in exp_placed:
                    visible |= p.sight().hexes
                exp_sight_sum += len(visible & land_set)
                exp_sight_samples += 1
            
            alive = {sq: Move() for sq in parts.squads if sq.king_alive()}
            if not alive:
                break
            
            try:
                summaries = tp.play_round(alive)
            except MoveError:
                errors += 1
                break
            
            for s in summaries:
                if s['squad'] == bal_squad.name:
                    bal_harvest_total += s.get('food_harvested', 0)
                    bal_drain_dealt += s.get('food_drain_dealt', 0)
                    bal_drain_received += s.get('food_drain_received', 0)
    finally:
        PURCHASE_OBJ.clear()
        PURCHASE_OBJ.update(old_obj)
    
    bal_net = bal_harvest_total + bal_drain_dealt - bal_drain_received
    exp_avg_sight = exp_sight_sum / max(1, exp_sight_samples)
    
    return {
        'bal_net': bal_net,
        'bal_harvest': bal_harvest_total,
        'bal_drain_dealt': bal_drain_dealt,
        'bal_drain_received': bal_drain_received,
        'exp_avg_sight': exp_avg_sight,
        'exp_max_sight': exp_sight_sum,  # cumulative
        'bal_alive': bal_squad.king_alive(),
        'exp_alive': exp_squad.king_alive(),
        'bal_placed': len([p for p in bal_squad.pieces if p.place is not None]),
        'exp_placed': len([p for p in exp_squad.pieces if p.place is not None]),
        'turns': tp.turn,
        'errors': errors,
    }
```

We need to refactor to handle pending

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Looking at `GlobalGameState`, the migration from individual pending fields to `PendingMove` is half-done. The issues:

1. **Commented-out fields** still in `GlobalGameState` (dead code)
2. **`place_piece` / `unplace_piece`** still reference `self.pending_placement` directly instead of `self.pendingMove`
3. **`selected_piece`** is duplicated — it exists on both `GS` and `PendingMove`

Here's the cleaned-up refactor:

```python
@dataclass
class GlobalGameState:
    parts: GameParts | None = None
    tp: TurnProcessor | None = None
    player_squad: Squad | None = None
    pendingMove: PendingMove | None = None
    active_overlays: set = field(default_factory=lambda: {'rivers','lanterns','network','ocean'})
    turn: int = 0
    radius: float = 25
    debug_overlays: bool = False
    rotationClickId: int = None
    ai_strategies: dict[str, PlacementStrategy] = field(default_factory=dict)
    focused_piece: Piece | None = None
    charts: TurnCharts | None = None

    def select_piece(self, piece: Piece):
        """Select an unplaced piece for placement."""
        if piece.place is not None:
            return  # already on board — rotation handled separately
        self.pendingMove.selected_piece = piece

    def place_piece(self, hex_idx: int) -> bool:
        """Place the selected piece at a hex."""
        pm = self.pendingMove
        if pm is None or pm.selected_piece is None:
            return False
        loc = self.parts.location_by_index(hex_idx)
        pm.set_placement(pm.selected_piece, loc)
        pm.selected_piece.place = loc   # preview immediately
        return True

    def unplace_piece(self, piece: Piece):
        """Remove a pending placement (before committing)."""
        pm = self.pendingMove
        if pm and pm.placement and pm.placement.piece is piece:
            piece.place = None
            pm.placement = None

    @property
    def is_active(self) -> bool:
        return self.parts is not None and self.tp is not None

    @property
    def game_over(self) -> bool:
        return self.tp.game_over if self.tp else False
    
    # Convenience pass-throughs so routes don't reach into pendingMove directly
    @property
    def selected_piece(self) -> Piece | None:
        return self.pendingMove.selected_piece if self.pendingMove else None
    
    @property
    def has_pending(self) -> bool:
        return self.pendingMove is not None and self.pendingMove.has_actions
```

Key changes:
- Removed `selected_piece` as a field — it's now a **property** delegating to `pendingMove`, so existing route code using `GS.selected_piece` keeps working
- `place_piece` / `unplace_piece` go through `pendingMove` properly
- Added `has_pending` convenience property for the UI "Commit" button enabled state

In [ ]:
@patch
def squad_flow(self: GlobalGameState, squad: Squad = None) -> SquadFlow:
    """Cached SquadFlow, invalidated on commit."""
    squad = squad or self.player_squad
    cache = getattr(self, '_flow_cache', {})
    key = (squad.name, self.turn)
    if key not in cache:
        cache[key] = SquadFlow.from_squad(squad)
        self._flow_cache = {key: cache[key]}  # keep only current turn
    return cache[key]


In [ ]:

@patch
def rotate_piece(self: GlobalGameState, piece: Piece):
    """Rotate a placed piece clockwise by one step."""
    if piece.place is None:
        return
    dirs = HexPosition.directions()
    cur_idx = dirs.index(piece.place.facing) if piece.place.facing in dirs else 0
    new_facing = dirs[(cur_idx + 1) % len(dirs)]
    
    # If this is the pending placement piece, just update its facing
    # (not a rotation — it hasn't been committed yet)
    if self.pendingMove.placement and self.pendingMove.placement.piece is piece:
        piece.place.facing = new_facing
        # Update the placement's location facing too
        self.pendingMove.placement.location.facing = new_facing
        return
    
    # Otherwise it's a real rotation of an already-placed piece
    self.pendingMove.rotations = [r for r in self.pendingMove.rotations if r.piece is not piece]
    self.pendingMove.rotations.append(Rotation(piece, new_facing))
    piece.place.facing = new_facing  # preview immediately


@patch
def commit_move(self: GlobalGameState) -> dict | None:
    if self.parts is None:
        return None
    
    # Undo preview placement so Move.from_squad sees clean state
    if self.pending_placement is not None:
        self.pending_placement.piece.place = None
    
    try:
        move = Move.from_squad(
            self.player_squad,
            rotations=self.pending_rotations,
            placement=self.pending_placement,
            purchase=self.pending_purchase)
    except MoveError as e:
        if self.pending_placement is not None:
            self.pending_placement.piece.place = self.pending_placement.location
        return {'error': str(e)}
    
    summary = self.tp.resolve_move(self.player_squad, move)
    
    # --- AI turns (matches simulation loop) ---
    ai_summaries = []
    for squad in self.parts.squads:
        if squad is self.player_squad or not squad.king_alive():
            continue
        old_strat = squad.strategy
        squad.reconsider(self.parts)
        squad.strategy = old_strat  # keep assigned strategy
        ai_move = squad.execute_queue(self.parts)
        if ai_move is not None:
            try:
                ai_summaries.append(self.tp.resolve_move(squad, ai_move))
            except MoveError:
                squad.queue.clear()
                squad.plan_ctx = None
    
    # --- End-of-turn food/drain resolution ---
    alive = {sq: Move() for sq in self.parts.squads if sq.king_alive()}
    if alive:
        try:
            round_summaries = self.tp.play_round(alive)
        except MoveError:
            round_summaries = []
    else:
        round_summaries = []
    
    self.tp.record.snapshot(self.turn, self.parts, player_squad=self.player_squad)
    self.turn += 1
    
    if self.turn % 3 == 0:
        for squad in self.parts.squads:
            squad.optimize_facings(self.parts)
    
    # Clear pending state
    self.pending_rotations = []
    self.pending_placement = None
    self.pending_purchase = None
    self.pending_purchase_type = None
    self.selected_piece = None
    
    return {'player': summary, 'ai': ai_summaries, 
            'round': round_summaries, 'turn': self.turn}


In [ ]:
@patch
def commit_move(self: GlobalGameState) -> dict | None:
    if self.parts is None: return None
    pm = self.pendingMove

    # Undo preview placement so Move.from_squad sees clean state
    if pm.placement is not None:
        pm.placement.piece.place = None

    try:
        move = pm.commit()  # delegates to Move.from_squad
    except MoveError as e:
        # Restore preview
        if pm.placement is not None:
            pm.placement.piece.place = pm.placement.location
        return {'error': str(e)}

    summary = self.tp.resolve_move(self.player_squad, move)

    # --- AI turns ---
    ai_summaries = []
    for squad in self.parts.squads:
        if squad is self.player_squad or not squad.king_alive():
            continue
        old_strat = squad.strategy
        squad.reconsider(self.parts)
        squad.strategy = old_strat
        ai_move = squad.execute_queue(self.parts)
        if ai_move is not None:
            try:
                ai_summaries.append(self.tp.resolve_move(squad, ai_move))
            except MoveError:
                squad.queue.clear()
                squad.plan_ctx = None

    # --- End-of-turn food/drain ---
    alive = {sq: Move() for sq in self.parts.squads if sq.king_alive()}
    round_summaries = []
    if alive:
        try:
            round_summaries = self.tp.play_round(alive)
        except MoveError:
            pass

    self.tp.record.snapshot(self.turn, self.parts, player_squad=self.player_squad)
    self.turn += 1

    if self.turn % 3 == 0:
        for squad in self.parts.squads:
            squad.optimize_facings(self.parts)

    # Clear via PendingMove
    pm.clear()

    return {'player': summary, 'ai': ai_summaries,
            'round': round_summaries, 'turn': self.turn}


We redid how strategies work and you can see how we did an optiixation run below. We need to refactor commit_move

```python
def run_expansive_trial(parts: GameParts,
                        exp_comp: dict[PieceType, int],
                        exp_weights: tuple[float, ...],
                        starting_pantry: int = 100,
                        seed: int = 42) -> dict:
    """Run expansive vs passive balanced. Measure territory coverage."""
    
    if not hasattr(parts, '_oasis_idxs'):
        parts._oasis_idxs = parts.fy.find_oasis(3, min_distance=6)
    
    parts.squads.clear()
    parts.loadSquads(year=1950, seed=seed, levels=2)
    for squad in parts.squads:
        for p in squad.pieces:
            p.place = None
    
    starts = parts.find_starts(n=2, ring_radius=3, min_distance=8)
    for squad, start_idx in zip(parts.squads, starts):
        king = next(p for p in squad.pieces if p.type == PieceType.KING)
        loc = parts.location_by_index(start_idx)
        Move.from_squad(squad, placement=Placement(king, loc)).apply(parts)
        king.place.pantry = starting_pantry
    
    exp_squad, bal_squad = parts.squads[0], parts.squads[1]
    
    exp_squad.strategy = PlacementStrategy(
        "expansive", defend_threat=3, attack_pantry=80, attack_placed=6,
        min_reserve_per_piece=4, min_budget_ratio=0.8, buy_cooldown=0,
        target_composition=exp_comp, graduation="aggressive")
    
    # Balanced opponent uses fixed known-good params — not tuned this run
    bal_squad.strategy = PlacementStrategy(
        "balanced", defend_threat=2, attack_pantry=60, attack_placed=4,
        min_reserve_per_piece=5, min_budget_ratio=1.0, buy_cooldown=0,
        target_composition={PieceType.PAWN: 4, PieceType.BISHOP: 2, 
                            PieceType.KNIGHT: 1, PieceType.QUEEN: 1},
        graduation="aggressive")
    
    trial_weights = {
        "expansive": {"expand": exp_weights, "defend": exp_weights, "attack": exp_weights},
        "balanced":  {"expand": (2.0,1.0,2.0,2.0,1.0),  # fixed
                      "defend": (2.0,1.0,3.0,1.0,3.0),
                      "attack": (1.0,3.0,1.5,1.0,1.0)},
    }
    
    tp = TurnProcessor(parts, parts.squads)
    for squad in parts.squads:
        squad.place_optimal_pawns(parts, tp, k=3)
    
    land_set = {i for i in range(len(parts.terr.elevations))
                if parts.terr.elevations[i] > 0}
    total_land = len(land_set)
    
    total_pieces = sum(exp_comp.values())
    max_rounds = total_pieces * 4 + 20
    
    sight_samples = []
    peak_sight = 0
    
    old_obj = dict(PURCHASE_OBJ)
    PURCHASE_OBJ.update(trial_weights)
    
    try:
        for turn in range(max_rounds):
            if tp.game_over: break
            
            for squad in parts.squads:
                if not squad.king_alive(): continue
                old_strat = squad.strategy
                squad.reconsider(parts)
                squad.strategy = old_strat
                move = squad.execute_queue(parts)
                if move is not None:
                    try: tp.resolve_move(squad, move)
                    except MoveError:
                        squad.queue.clear()
                        squad.plan_ctx = None
            
            # Measure expansive sight
            exp_placed = [p for p in exp_squad.pieces if p.place is not None]
            if exp_placed:
                visible = set()
                for p in exp_placed: visible |= p.sight().hexes
                coverage = len(visible & land_set)
                sight_samples.append(coverage)
                peak_sight = max(peak_sight, coverage)
            
            alive = {sq: Move() for sq in parts.squads if sq.king_alive()}
            if not alive: break
            try: tp.play_round(alive)
            except MoveError: break
    finally:
        PURCHASE_OBJ.clear()
        PURCHASE_OBJ.update(old_obj)
    
    exp_king = next((p for p in exp_squad.pieces 
                     if p.type == PieceType.KING and p.place is not None), None)
    
    return {
        'avg_sight': np.mean(sight_samples) if sight_samples else 0,
        'peak_sight': peak_sight,
        'pct_land_covered': peak_sight / max(1, total_land),
        'exp_alive': exp_squad.king_alive(),
        'exp_placed': len([p for p in exp_squad.pieces if p.place is not None]),
        'exp_pantry': exp_king.place.pantry if exp_king else 0,
        'turns': tp.turn,
    }

def objective_expansive(trial: optuna.Trial) -> tuple[float, float]:
    """Maximize avg sight coverage AND peak sight."""
    
    exp_pawns   = trial.suggest_int("exp_pawns", 3, 6)
    exp_bishops = trial.suggest_int("exp_bishops", 3, 7)  # bishops are the point
    exp_knights = trial.suggest_int("exp_knights", 0, 2)
    exp_queens  = trial.suggest_int("exp_queens", 0, 1)
    exp_comp = {PieceType.PAWN: exp_pawns, PieceType.BISHOP: exp_bishops,
                PieceType.KNIGHT: exp_knights, PieceType.QUEEN: exp_queens}
    
    # Weights: food, drain, connect, frontier, threat
    # For expansive, frontier (w3) should matter most
    exp_w = tuple(trial.suggest_float(f"exp_w{i}", 0.1, 5.0) for i in range(5))
    
    avg_sights, peak_sights = [], []
    for seed in range(5):
        result = run_expansive_trial(board, exp_comp, exp_w, seed=seed)
        avg_sights.append(result['avg_sight'])
        peak_sights.append(result['peak_sight'])
    
    return np.mean(avg_sights), np.mean(peak_sights)

study_exp = optuna.create_study(
    directions=["maximize", "maximize"],
    study_name="expansive_v1",
)
study_exp.optimize(objective_expansive, n_trials=5)
```

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The main differences between `commit_move` and the simulation loop:

1. **AI move generation** — `commit_move` calls `strategy.next_move(squad, parts)` which doesn't exist in the simulation. The simulation uses `squad.reconsider(parts)` → `squad.execute_queue(parts)`.

2. **No `play_round()`** — the simulation calls `tp.play_round(alive)` after all moves to resolve food/drain. `commit_move` skips this entirely.

3. **No `PURCHASE_OBJ` weights** — the simulation manages per-strategy weights.

4. **No error recovery** — the simulation clears `squad.queue` and `squad.plan_ctx` on `MoveError`.

5. **Strategy stored in wrong place** — `ai_strategies` dict duplicates what should live on `squad.strategy`.

Here's the refactored version:

```python
@patch
def commit_move(self: GlobalGameState) -> dict | None:
    if self.parts is None:
        return None
    
    # Undo preview placement so Move.from_squad sees clean state
    if self.pending_placement is not None:
        self.pending_placement.piece.place = None
    
    try:
        move = Move.from_squad(
            self.player_squad,
            rotations=self.pending_rotations,
            placement=self.pending_placement,
            purchase=self.pending_purchase)
    except MoveError as e:
        if self.pending_placement is not None:
            self.pending_placement.piece.place = self.pending_placement.location
        return {'error': str(e)}
    
    summary = self.tp.resolve_move(self.player_squad, move)
    
    # --- AI turns (matches simulation loop) ---
    ai_summaries = []
    for squad in self.parts.squads:
        if squad is self.player_squad or not squad.king_alive():
            continue
        old_strat = squad.strategy
        squad.reconsider(self.parts)
        squad.strategy = old_strat  # keep assigned strategy
        ai_move = squad.execute_queue(self.parts)
        if ai_move is not None:
            try:
                ai_summaries.append(self.tp.resolve_move(squad, ai_move))
            except MoveError:
                squad.queue.clear()
                squad.plan_ctx = None
    
    # --- End-of-turn food/drain resolution ---
    alive = {sq: Move() for sq in self.parts.squads if sq.king_alive()}
    if alive:
        try:
            round_summaries = self.tp.play_round(alive)
        except MoveError:
            round_summaries = []
    else:
        round_summaries = []
    
    self.tp.record.snapshot(self.turn, self.parts, player_squad=self.player_squad)
    self.turn += 1
    
    if self.turn % 3 == 0:
        for squad in self.parts.squads:
            squad.optimize_facings(self.parts)
    
    # Clear pending state
    self.pending_rotations = []
    self.pending_placement = None
    self.pending_purchase = None
    self.pending_purchase_type = None
    self.selected_piece = None
    
    return {'player': summary, 'ai': ai_summaries, 
            'round': round_summaries, 'turn': self.turn}
```

You'll also want to update `start_game` to set strategies directly on `squad.strategy` instead of the separate `ai_strategies` dict — that way the strategies live where the simulation expects them. The `ai_strategies` field on `GlobalGameState` can then be removed.

can you refactor start_game

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

```python
@patch
def start_game(self: GlobalGameState, size=300, player_count=3, seed=23, radius=20, debug=False):
    random.seed(seed)
    np.random.seed(seed)
    starting_pantry = 100

    world = TerrainFactory.create_world(
        bounds=MapRect(MapCord(0, 0), MapSize(size, size)),
        preset='temperate', name='Maiden Lane', radius=radius,
        lon_span=10.0, num_plates=8, subdivisions=3,
        ocean_fraction=0.1, oceanic_sides=['N'],
        terrain_age='young', formation_type='ridge',
        elevation_scale=1.5, erosion_age=0.1, num_lakes=0,
        seed=seed, debug=debug)

    self.parts = GameParts(world)
    if debug:
        self.parts.terr.__ft__()
        self.parts.fy.summary()
        show(self.parts.terr)
    self.parts.grid.adjustRadius(radius)

    # --- Squad creation ---
    starts = self.parts.find_starts(n=player_count, ring_radius=3, min_distance=9)
    self.parts.flags = CountryFlag.seaborn("bright", levels=player_count)
    year = 1880

    for i, flag in enumerate(self.parts.flags):
        squad = flag.make_squad([PieceType.KING], year=year)
        king = next(p for p in squad.pieces if p.type == PieceType.KING)
        loc = self.parts.location_by_index(starts[i])
        Move.from_squad(squad, placement=Placement(king, loc)).apply(self.parts)
        king.place.pantry = starting_pantry
        self.parts.squads.append(squad)

    # --- TurnProcessor must exist before placing pawns ---
    self.tp = TurnProcessor(self.parts, self.parts.squads)

    for sq in self.parts.squads:
        sq.place_optimal_pawns(self.parts, self.tp, k=3)

    # --- Assign strategies directly on squads ---
    self.player_squad = self.parts.squads[0]
    self.player_squad.strategy = PlacementStrategy.BALANCED

    ai_pool = [PlacementStrategy.AGGRESSIVE, PlacementStrategy.EXPANSIVE,
               PlacementStrategy.TURTLE, PlacementStrategy.BALANCED]
    for i, sq in enumerate(self.parts.squads[1:]):
        sq.strategy = ai_pool[i % len(ai_pool)]

    # --- Clear pending state ---
    self.selected_piece = None
    self.pending_rotations = []
    self.pending_placement = None
    self.pending_purchase = None
    self.pending_purchase_type = None
    self.active_chart_cols = {'king_pantry', 'pieces_placed'}
    self.active_chart_squads = set()
    self.turn = 0
```

Key changes:

- **Explicit king placement via `Move.from_squad().apply()`** — matches the simulation pattern exactly, so validation runs the same path as gameplay
- **`place_optimal_pawns` called after `TurnProcessor` creation** — was missing entirely before
- **Strategies set on `squad.strategy` directly** — eliminates the `ai_strategies` dict (you can remove that field from `GlobalGameState`)
- **Player squad gets `BALANCED`** — explicit rather than implicit
- **Removed the `setup_game()` black box** — full control over initialization order

In [ ]:
@patch
def move_panel(gs: GlobalGameState) -> FT:
    gs.pendingMove.squad = gs.player_squad
    return gs.parts.movePanel(gs.pendingMove)


In [ ]:
@patch
def _build_overlays(GS:GlobalGameState):
    """Assemble overlay list from GS.active_overlays."""
    if 'terrain' in GS.active_overlays:
        overlays = [TerrainOverlay(), RiverOverlay(max_width=4)]
    else:
        overlays = [CreamOverlay(stylized=True), RiverOverlay(max_width=4)]
    
    OVERLAY_MAP = {
        'food':          FoodOverlay,
        'lanterns':      LanternOverlay,
        'network':       FoodNetworkOverlay,
        'optimal':       lambda: OptimalPlacementOverlay(top_n=3),
        'ocean':         OceanWaveOverlay,
        'number':         HexNumberOverlay,
        'soil':          SoilOverlay,
        'climate':       ClimateOverlay,
        'temperature':   TemperatureOverlay,
        'precipitation': PrecipitationOverlay,
    }
    for key, factory in OVERLAY_MAP.items():
        if key in GS.active_overlays:
            overlays.append(factory())
    
    overlays.append(FogOverlay())
    overlays.append(GS.RotationTargetOverlay())
    overlays.append(PieceOverlay())
    if 'names' in GS.active_overlays:
        overlays.append(SquadNamesOverlay())
    
    return overlays

@patch
def _focus_bar(GS:GlobalGameState):
    """Focus banner + region if a piece is focused, else (None, "")."""
    focused = GS.focused_piece
    if focused and focused.place is not None:
        region = focused.sight()
        bar = DivFullySpaced(
            DivLAligned(
                Span(focused.type.icon),
                Strong(focused.name, cls="text-sm"),
                Small(f"Hex {GS.parts.hp2i(focused.place.location)}",
                      cls=TextPresets.muted_sm),
                Small(f"👁️ {len(region.hexes)} hexes", cls=TextPresets.muted_sm),
            ),
            Button("✕ Full Map", cls=(ButtonT.ghost, "text-xs"),
                   hx_post="/piece/unfocus", hx_target="#center-panel"),
            cls="px-3 py-1 bg-primary/10 rounded-md mb-2",
        )
        return region, bar
    return None, ""

In [ ]:
@patch
def _placement_status(GS: GlobalGameState):
    """Status line showing pending selection/placement."""
    pm = GS.pendingMove

    if GS.rotationClickId is not None:
        piece = next((p for p in GS.player_squad.pieces
                      if p.place is not None
                      and GS.parts.hp2i(p.place.location) == GS.rotationClickId), None)
        name = f"{piece.type.icon} {piece.name}" if piece else "piece"
        return P(f"⟳ Click a direction to rotate {name}",
                 cls="text-primary font-medium text-center")

    if pm.selected_piece is not None:
        p = pm.selected_piece
        return P(f"📍 Click map to place {p.type.icon} {p.name}",
                 cls="text-primary font-medium text-center")

    if pm.placement is not None:
        p = pm.placement.piece
        return P(f"Pending: {p.type.icon} {p.name} → hex. Commit or click to undo.",
                 cls="text-amber-600 text-center text-sm")

    if pm.purchase_type is not None:
        return P(f"📍 Click map to place new {pm.purchase_type.value.title()}",
                 cls="text-primary font-medium text-center")

    return ""


need to refactor _placement_status

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The issue is it's reading `GS.pending_placement` and `GS.selected_piece` directly instead of going through `GS.pendingMove`:

```python
@patch
def _placement_status(GS: GlobalGameState):
    """Status line showing pending selection/placement."""
    pm = GS.pendingMove

    if GS.rotationClickId is not None:
        piece = next((p for p in GS.player_squad.pieces
                      if p.place is not None
                      and GS.parts.hp2i(p.place.location) == GS.rotationClickId), None)
        name = f"{piece.type.icon} {piece.name}" if piece else "piece"
        return P(f"⟳ Click a direction to rotate {name}",
                 cls="text-primary font-medium text-center")

    if pm.selected_piece is not None:
        p = pm.selected_piece
        return P(f"📍 Click map to place {p.type.icon} {p.name}",
                 cls="text-primary font-medium text-center")

    if pm.placement is not None:
        p = pm.placement.piece
        return P(f"Pending: {p.type.icon} {p.name} → hex. Commit or click to undo.",
                 cls="text-amber-600 text-center text-sm")

    if pm.purchase_type is not None:
        return P(f"📍 Click map to place new {pm.purchase_type.value.title()}",
                 cls="text-primary font-medium text-center")

    return ""
```

Changes:
- `GS.selected_piece` → `pm.selected_piece`
- `GS.pending_placement` → `pm.placement` (using `PendingMove` field names)
- Added a `purchase_type` branch — when a piece type is selected but not yet placed on the map, the user needs a prompt too (previously unhandled)

In [ ]:
@patch
def panel_map_content(GS:GlobalGameState):
    """Map view — zoomed to focused piece or full map."""
    overlays = GS._build_overlays()
    region, focus_bar = GS._focus_bar()
    
    ctx = GS.parts.overlayContext(
        region=region, padding=2 if region else 0, radius=GS.radius)
    ctx.grid.builder.layers = []
    ctx.grid.adjustRadius(GS.radius)
    TerrainDisplay(*overlays, radius=GS.radius, ctx=ctx, debug=GS.debug_overlays)
    
    map_div = HexTouchMap(
        ctx.grid,
        on_click=HexWrapper.route("/hex_click", "#center-panel"),
        cls="w-full h-full",
    )
    
    return Div(focus_bar, map_div, GS._placement_status(), GS._map_controls())

In [ ]:
@patch
def panel_flow(GS:GlobalGameState):
    """Flow network diagram — standalone tab."""
    squad = GS.player_squad
    placed = [p for p in squad.pieces if p.place is not None]
    if len(placed) < 2:
        return P("Need at least 2 placed pieces to show flow.", cls=TextPresets.muted_sm)
    
    return SquadFlowDiagram(
        squad=squad, parts=GS.parts,
        width=700, height=450,
        on_click=lambda p: {
            "hx-post": "/piece/focus",
            "hx-vals": json.dumps({"piece_id": p.id}),
            "hx-target": "#center-panel",
        },
    ).__ft__()

In [ ]:
@patch
def panel_squad_detail(GS:GlobalGameState):
    """Squad detail view — reuses squad table + network info."""
    squad = GS.player_squad
    
    # Squad table (reuse existing .table() method)
    table_card = squad.table()
    diag = SquadFlowDiagram(squad=squad, parts=GS.parts,
                            width=600, height=300)
    
    deadcode = """
    # Network health summary
    G = squad.to_nx()
    U = G.to_undirected(as_view=True) if len(G) > 1 else None
    bottlenecks = list(nx.articulation_points(U)) if U else []
    isolated = squad.isolated_pieces()
    sources = squad.food_sources()
    sinks = squad.food_sinks()
    
    king = next((p for p in squad.pieces 
                 if p.type == PieceType.KING and p.place is not None), None)
    
    network_card = Card(
        Grid(
            Div(
                DivLAligned(Span("🔗", cls=TextT.lg), Strong("Connected")),
                P(f"{len(G.nodes)} pieces", cls=TextPresets.muted_sm),
            ),
            Div(
                DivLAligned(Span("⚠️", cls=TextT.lg), Strong("Bottlenecks")),
                P(f"{len(bottlenecks)}", cls=TextPresets.muted_sm),
            ),
            Div(
                DivLAligned(Span("🌾", cls=TextT.lg), Strong("Sources")),
                P(f"{len(sources)} pieces", cls=TextPresets.muted_sm),
            ),
            Div(
                DivLAligned(Span("🍽️", cls=TextT.lg), Strong("Sinks")),
                P(f"{len(sinks)} pieces", cls=TextPresets.muted_sm),
            ),
            cols_sm=2, cols_md=4,
        ),
        header=DivFullySpaced(
            H4("Network Health"),
            Label(f"🍞 {king.place.pantry if king else 0}", cls=LabelT.primary),
        ),
    )
    
    # Isolated pieces warning
    warnings = []
    if isolated:
        names = ", ".join(f"{p.type.icon} {p.name}" for p in isolated)
        warnings.append(P(f"⚠️ Isolated: {names}", cls="text-destructive text-sm"))
    if bottlenecks:
        names = ", ".join(f"{G.nodes[n]['piece'].type.icon} {G.nodes[n]['piece'].name}" 
                          for n in bottlenecks)
        warnings.append(P(f"🔗 Bottlenecks: {names}", cls="text-warning text-sm"))
    """
    return Div(diag, squad, cls="space-y-3")

In [ ]:
@patch
def _chart_controls(GS:GlobalGameState):
    """Two toggle rows: columns and squads."""
    # Row 1: metric columns
    col_toggles = DivLAligned(
        *[Button(label,
            cls=(ButtonT.primary if key in GS.charts.active_cols else ButtonT.ghost, "text-xs"),
            hx_post="/toggle_chart_col",
            hx_vals=json.dumps({"col": key}),
            hx_target="#chart-panel")
          for key, label in CHART_COLS],
        cls="gap-1 flex-wrap",
    )

    # Row 2: squads (flag color dots + name)
    squad_toggles = []
    all_squads = GS.parts.squads if GS.parts else []
    active = GS.charts.active_squads or {s.name for s in all_squads}  # empty = all shown

    for sq in all_squads:
        is_on = sq.name in active or not GS.charts.active_squads
        is_player = sq is GS.player_squad
        label = DivLAligned(
            Span("●", style=f"color:{sq.flag.primary}; font-size:1.1em"),
            Small(sq.name.split()[0], cls="font-medium text-xs" if is_player else "text-xs"),
        )
        squad_toggles.append(
            Button(label,
                cls=(ButtonT.primary if is_on else ButtonT.ghost, "text-xs px-2"),
                hx_post="/toggle_chart_squad",
                hx_vals=json.dumps({"squad": sq.name}),
                hx_target="#chart-panel")
        )

    return Div(
        DivLAligned(*squad_toggles, cls="gap-1 flex-wrap"),
        col_toggles,
        cls="space-y-1 pb-2 border-b",
    )

@patch
def panel_charts(GS:GlobalGameState):
    """Time-series sparklines — controlled by active_chart_cols + active_chart_squads."""
    if GS.tp is None or GS.tp.record.df.empty:
        return Div(
            _chart_controls(),
            P("No history yet — complete a turn.", cls=TextPresets.muted_sm),
            id="chart-panel", cls="space-y-3",
        )

    df = GS.tp.record.df
    all_squads = GS.parts.squads
    
    # Which squads to plot
    active_squads = GS.charts.active_squads or {s.name for s in all_squads}
    colors = {sq.name: sq.flag.primary for sq in all_squads}
    
    cols = sorted(GS.charts.active_cols)
    n = len(cols)
    ncols = min(3, n)
    nrows = math.ceil(n / ncols)
    
    fig, axes = plt.subplots(nrows, ncols,
                              figsize=(4.5 * ncols, 3.2 * nrows),
                              facecolor='none', squeeze=False)
    fig.patch.set_alpha(0)

    col_labels = dict(CHART_COLS)

    for i, col in enumerate(cols):
        ax = axes[i // ncols][i % ncols]
        
        for sq in all_squads:
            if sq.name not in active_squads:
                continue
            sub = df[df.squad == sq.name].sort_values('turn')
            if sub.empty or col not in sub.columns:
                continue
            is_player = sq is GS.player_squad
            is_alive = sub.iloc[-1]['alive']
            color = colors[sq.name]
            ax.plot(sub['turn'], sub[col],
                    color=color,
                    lw=2.5 if is_player else 1.2,
                    ls='-' if is_alive else '--',
                    alpha=1.0 if is_player else (0.55 if is_alive else 0.2),
                    marker='o', markersize=3 if is_player else 1.5,
                    label=sq.name.split()[0])
            if is_player and not sub.empty:
                last = sub.iloc[-1]
                ax.annotate(f"{last[col]:.0f}",
                            xy=(last['turn'], last[col]),
                            fontsize=7, color=color,
                            xytext=(4, 0), textcoords='offset points')

        ax.set_title(col_labels.get(col, col), fontsize=10, fontweight='bold')
        ax.tick_params(labelsize=7)
        ax.grid(alpha=0.2, lw=0.5)
        ax.spines[['top', 'right']].set_visible(False)
        ax.set_xlabel("Turn", fontsize=7)

    # Hide unused axes
    for j in range(n, nrows * ncols):
        axes[j // ncols][j % ncols].set_visible(False)

    # Shared legend
    handles, labels = axes[0][0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc='lower center',
                   ncol=len(active_squads), fontsize=8,
                   framealpha=0.3, bbox_to_anchor=(0.5, -0.02))

    plt.tight_layout(rect=[0, 0.06, 1, 1])

    buf = io.BytesIO()
    fig.savefig(buf, format='svg', bbox_inches='tight', transparent=True)
    plt.close(fig)
    buf.seek(0)

    return Div(
        GS._chart_controls(),
        NotStr(buf.read().decode()),
        id="chart-panel",
        cls="space-y-2",
    )


In [ ]:
@patch
def RotationTargetOverlay(GS:GlobalGameState,**kw) -> OverlaySpec:
    """When in rotation mode, highlight 6 adjacent hexes with direction labels."""
    def render(ctx) -> str:
        if GS.rotationClickId is None:
            return ""
        
        grid = ctx.grid
        N = len(grid.hexes)
        parts_list = []
        
        src_idx = GS.rotationClickId
        src_hp = GS.parts.i2hp(src_idx)
        
        # Find the piece being rotated (for its current facing)
        piece = next((p for p in GS.player_squad.pieces
                     if p.place is not None
                     and GS.parts.hp2i(p.place.location) == src_idx), None)
        current_facing = piece.place.facing if piece else None
        
        for d in HexPosition.directions():
            neighbor_hp = src_hp + d
            neighbor_idx = GS.parts.hp2i(neighbor_hp)
            
            for fi in ctx.fine_indices(neighbor_idx):
                if 0 <= fi < N:
                    h = grid.hexes[fi]
                    c = h.center
                    pts = " ".join(f"{p.x:.0f},{p.y:.0f}" for p in h.vertices())
                    
                    # Highlight: brighter for current facing
                    is_current = (d == current_facing)
                    fill_color = "#FFD700" if is_current else "#4a90d9"
                    opacity = 0.5 if is_current else 0.3
                    
                    parts_list.append(
                        f'<polygon points="{pts}" '
                        f'fill="{fill_color}" fill-opacity="{opacity}" '
                        f'stroke="{fill_color}" stroke-width="2" stroke-opacity="0.7"/>')
                    
                    # Direction label
                    fs = max(10, grid.radius * 0.4)
                    parts_list.append(
                        f'<text x="{c.x:.1f}" y="{c.y:.1f}" '
                        f'text-anchor="middle" dominant-baseline="central" '
                        f'font-size="{fs:.0f}" font-weight="bold" '
                        f'fill="white" opacity="0.9">{d.label}</text>')
        
        return '\n'.join(parts_list)
    
    return OverlaySpec("rotation_targets", render, priority=92)


In [ ]:
def appRoutes():
    global app, rt, hexserver
    #if 'hexserver' not in dir() or hexserver is None:
    if 'hexserver' not in globals() or hexserver is None:

        app = FastHTML(hdrs=Theme.violet.headers(), secret_key=SECRECT_KEY)
        rt = app.route
        hexserver = JupyUvi(app)
    return app, rt, hexserver

In [ ]:
app, rt,  hexserver = appRoutes()

In [ ]:
game_states: dict[str, GlobalGameState] = {}


In [ ]:

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s')
log = logging.getLogger("hexgame")

# In-memory log ring buffer
game_log = deque(maxlen=50)

class GameLogHandler(logging.Handler):
    def emit(self, record):
        game_log.append(self.format(record))

log.addHandler(GameLogHandler())
log.setLevel(logging.INFO)

# Option 1: Attach to uvicorn loggers directly
for name in ('uvicorn', 'uvicorn.access', 'uvicorn.error'):
    logging.getLogger(name).addHandler(GameLogHandler())


read_url("https://fastht.ml/docs/llms-ctx.txt")

I need something like the below. but instead of having a globalgamestate called GS, we need to look one up based upon user session

```python
# ── Landing page ──
@rt
def index():
    return Title("HexMagic"), Container(
        DivCentered(
            H1("⬡ HexMagic", cls="uk-h1"),
            P("Hex-based strategy with food economics", cls=TextPresets.muted_lg),
            Card(
                Form(
                    Grid(
                        LabelInput("Map Size", id="size", type="number", 
                                   value="300", placeholder="200-600"),
                        LabelInput("Players", id="player_count", type="number",
                                   value="3", placeholder="2-5"),
                        LabelInput("Seed", id="seed", type="number",
                                   value="23", placeholder="Any integer"),
                        LabelInput("Hex Radius", id="radius", type="number",
                                   value="20", placeholder="15-30"),
                        cols_sm=2,
                    ),
                    Button("⬡ Start Game", cls=(ButtonT.primary, "w-full mt-4"),
                           hx_post="/start_game", hx_target="body"),
                ),
                header=(H3("New Game"), Subtitle("Configure your world")),
            ),
            cls="min-h-screen py-12",
        )
    )

# ── Start game ──
@rt
def start_game(size: int = 300, player_count: int = 3, 
               seed: int = 23, radius: int = 20):
    player_count = max(2, min(5, player_count))
    size = max(200, min(600, size))
    radius = max(10, min(40, radius))
    
    GS.start_game(size=size, player_count=player_count, 
                   seed=seed, radius=radius)
    
    return RedirectResponse("/game", status_code=303)


```

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

A common approach is to keep a dict of game states keyed by a session ID. In FastHTML, you can access the session via the `session` parameter in your route handlers.

Here's the core idea:

```python
import uuid

game_states = {}  # session_id -> GameState

def get_gs(session):
    sid = session.get("sid")
    if sid and sid in game_states:
        return game_states[sid]
    return None
```

Then in `start_game`, you'd create a new session ID and game state:

```python
def start_game(session, size: int = 300, ...):
    sid = str(uuid.uuid4())
    session["sid"] = sid
    gs = GameState()
    gs.start_game(size=size, ...)
    game_states[sid] = gs
    return RedirectResponse("/game", status_code=303)
```

And in any other route, just call `get_gs(session)` to retrieve it.

A couple of things to consider before going further:

1. **Persistence** — this dict is in-memory only, so game states are lost on server restart. Is that OK for now?
2. **Cleanup** — do you need to handle expired/abandoned sessions?

What's your situation on those?

Just persistence for now.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Perfect, in-memory is fine for a first pass. The pattern I showed should work as-is then — just make sure `game_states` is defined at the module level so it persists across requests.

One small thing to be aware of: if you're running with multiple workers (e.g. `uvicorn --workers 4`), each worker gets its own `game_states` dict, so sessions could break across requests. Running with a **single worker** avoids that for now.

Want to go ahead and wire it up, or would you like a quick sketch of how the `/game` route would look with session lookup too?

In [ ]:
import uuid

game_states = {}  # session_id -> GameState

def get_gs(session):
    sid = session.get("sid")
    if sid and sid in game_states:
        return game_states[sid]
    return None

def get_gs_or_redirect(session):
    gs = get_gs(session)
    if gs is None:
        return None, RedirectResponse("/", status_code=303)
    return gs, None


In [ ]:
# ── Landing page ──
@rt
def index():
    return Title("HexMagic"), Container(
        DivCentered(
            H1("⬡ HexMagic", cls="uk-h1"),
            P("Hex-based strategy with food economics", cls=TextPresets.muted_lg),
            Card(
                Form(
                    Grid(
                        LabelInput("Map Size", id="size", type="number", 
                                   value="300", placeholder="200-600"),
                        LabelInput("Players", id="player_count", type="number",
                                   value="3", placeholder="2-5"),
                        LabelInput("Seed", id="seed", type="number",
                                   value="23", placeholder="Any integer"),
                        LabelInput("Hex Radius", id="radius", type="number",
                                   value="20", placeholder="15-30"),
                        cols_sm=2,
                    ),
                    Button("⬡ Start Game", cls=(ButtonT.primary, "w-full mt-4"),
                           hx_post="/start_game", hx_target="body"),
                ),
                header=(H3("New Game"), Subtitle("Configure your world")),
            ),
            cls="min-h-screen py-12",
        )
    )


In [ ]:
@patch
def start_game(self: GlobalGameState, size=300, player_count=3, seed=23, radius=20, debug=False):
    random.seed(seed)
    np.random.seed(seed)
    starting_pantry = 100

    world = TerrainFactory.create_world(
        bounds=MapRect(MapCord(0, 0), MapSize(size, size)),
        preset='temperate', name='Maiden Lane', radius=radius,
        lon_span=10.0, num_plates=8, subdivisions=3,
        ocean_fraction=0.1, oceanic_sides=['N'],
        terrain_age='young', formation_type='ridge',
        elevation_scale=1.5, erosion_age=0.1, num_lakes=0,
        seed=seed, debug=debug)

    self.parts = GameParts(world)
    if debug:
        self.parts.terr.__ft__()
        self.parts.fy.summary()
        show(self.parts.terr)
    self.parts.grid.adjustRadius(radius)

    # --- Squad creation ---
    starts = self.parts.find_starts(n=player_count, ring_radius=3, min_distance=9)
    self.parts.flags = CountryFlag.seaborn("bright", levels=player_count)
    year = 1880

    for i, flag in enumerate(self.parts.flags):
        squad = flag.make_squad([PieceType.KING], year=year)
        king = next(p for p in squad.pieces if p.type == PieceType.KING)
        loc = self.parts.location_by_index(starts[i])
        Move.from_squad(squad, placement=Placement(king, loc)).apply(self.parts)
        king.place.pantry = starting_pantry
        self.parts.squads.append(squad)

    # --- TurnProcessor must exist before placing pawns ---
    self.tp = TurnProcessor(self.parts, self.parts.squads)

    for sq in self.parts.squads:
        sq.place_optimal_pawns(self.parts, self.tp, k=3)

    # --- Assign strategies directly on squads ---
    self.player_squad = self.parts.squads[0]
    self.player_squad.strategy = PlacementStrategy.BALANCED

    ai_pool = [PlacementStrategy.AGGRESSIVE, PlacementStrategy.EXPANSIVE,
               PlacementStrategy.TURTLE, PlacementStrategy.BALANCED]
    for i, sq in enumerate(self.parts.squads[1:]):
        sq.strategy = ai_pool[i % len(ai_pool)]

    # --- Clear pending state ---
        # --- Clear pending state ---
    self.pendingMove = PendingMove(squad=self.player_squad)
    self.charts = TurnCharts(tp=self.tp, player_squad=self.player_squad)
    self.turn = 0



In [ ]:
@rt("/start_game", methods=["POST"])
def new_game(session, size: int = 300, player_count: int = 3,
               seed: int = 23, radius: int = 20):
    player_count = max(2, min(5, player_count))
    size        = max(200, min(600, size))
    radius      = max(10,  min(40,  radius))

    sid = str(uuid.uuid4())
    session["sid"] = sid

    gs = GlobalGameState()
    gs.start_game(size=size, player_count=player_count, seed=seed, radius=radius)
    game_states[sid] = gs

    return RedirectResponse("/game", status_code=303)

So I need at rt for start_game

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

```python
@rt
def start_game(session, size: int = 300, player_count: int = 3,
               seed: int = 23, radius: int = 20):
    player_count = max(2, min(5, player_count))
    size        = max(200, min(600, size))
    radius      = max(10,  min(40,  radius))

    sid = str(uuid.uuid4())
    session["sid"] = sid

    gs = GlobalGameState()
    gs.start_game(size=size, player_count=player_count, seed=seed, radius=radius)
    game_states[sid] = gs

    return RedirectResponse("/game", status_code=303)
```

Note the name conflict — `start_game` is both a `@patch` method on `GlobalGameState` **and** this route. FastHTML names routes from the function name, so you'll want to rename one of them. The cleanest fix is to rename the route function:

```python
@rt("/start_game", methods=["POST"])
def new_game(session, size: int = 300, ...):
    ...
```

In [ ]:
testGS = GlobalGameState()
testGS.start_game()

@rt("/commit", methods=["POST"])
def commit(session):
    GS, redir = get_gs_or_redirect(session)
    if redir: return redir
    result = GS.commit_move()
    if result and 'error' in result:
        return Div(
            P(f"⚠️ {result['error']}", cls="text-destructive font-medium text-center py-4"),
            Button("Back", hx_get="/game", hx_target="body"),
        )
    
    # Check if player was eliminated
    if result and result['player'].get('eliminated') == GS.player_squad.name:
        return Title("Eliminated"), Container(
            DivCentered(
                H2("☠️ Your Kingdom Falls", cls="text-destructive"),
                P(f"The treasury ran dry on turn {GS.turn}.", cls=TextPresets.muted_lg),
                _game_over_summary(),
                Button("🏠 New Game", cls=(ButtonT.primary, "mt-4"),
                       hx_get="/", hx_target="body"),
            )
        )
    
    # Check if game over (player won)
    if GS.tp.game_over:
        winner = GS.tp.winner
        if winner and winner is GS.player_squad:
            return Title("Victory"), Container(
                DivCentered(
                    H2("👑 Victory!", cls="text-yellow-500"),
                    P(f"{winner.name} is the last kingdom standing!", cls=TextPresets.muted_lg),
                    _game_over_summary(session),
                    Button("🏠 New Game", cls=(ButtonT.primary, "mt-4"),
                           hx_get="/", hx_target="body"),
                )
            )
    
    return RedirectResponse("/game", status_code=303)

In [ ]:



def _game_over_summary(session):
    """Final status table for end-of-game screens."""
    GS, redir = get_gs_or_redirect(session)
    if redir: return ""

    if not GS.tp: return ""
    return Card(
        Table(
            Thead(Tr(Th(""), Th("Kingdom"), Th("🍞 Pantry"), Th("Pieces"), Th("Turns"))),
            Tbody(*[Tr(
                Td("👑" if sq.king_alive() else "💀"),
                Td(sq.name, cls="font-medium"),
                Td(str(next((p.place.pantry for p in sq.pieces 
                             if p.type == PieceType.KING and p.place), 0))),
                Td(str(sum(1 for p in sq.pieces if p.place is not None))),
                Td(str(GS.turn)),
            ) for sq in GS.parts.squads]),
        ),
        header=H4(f"Final Standings — Turn {GS.turn}"),
    )

def gs_left_panel(gs: GlobalGameState) -> FT:
    """Left sidebar: purchase selection + pending move summary + commit/clear."""
    sq = gs.player_squad
    king = next((p for p in sq.pieces if p.type == PieceType.KING and p.place), None)
    king_pantry = king.place.pantry if king else 0

    sections = [PurchasePanel(sq.flag, king_pantry, gs.pending_purchase_type)]

    # Show pending state
    if gs.pending_purchase:
        pt = gs.pending_purchase.piece_type
        idx = gs.parts.hp2i(gs.pending_purchase.location.location)
        sections.append(Div(
            Divider(),
            P(f"📍 {pt.value.title()} → Hex {idx}", cls="text-sm font-medium"),
        ))
    if gs.pending_rotations:
        rot_lines = [f"⟳ {r.piece.name}" for r in gs.pending_rotations]
        sections.append(Div(
            Divider(),
            *[P(line, cls="text-sm") for line in rot_lines],
        ))

    # Commit / Clear buttons
    has_pending = gs.pending_purchase or gs.pending_rotations or gs.pending_placement
    sections.append(Div(
        Divider(),
        DivFullySpaced(
            Button("Clear", cls=ButtonT.ghost,
                   hx_post="/move/clear", hx_target="#move-panel", hx_swap="outerHTML"),
            Button("✅ Commit", cls=ButtonT.primary,
                   hx_post="/move/commit", hx_target="body",
                   disabled=not has_pending),
        ),
    ))

    return Div(*sections, id="move-panel", cls="p-4 space-y-3")


In [ ]:
@rt("/hex_click", methods=["POST"])
def hex_click(session, hex_id: int):
    GS, redir = get_gs_or_redirect(session)
    if redir: return redir
    log.info(f"HEX_CLICK hex_id={hex_id} rotationMode={GS.rotationClickId}")

    def center():  return Div(GS.panel_map_content(), id="center-panel")
    def left_oob(): return Div(GS.move_panel(), hx_swap_oob="outerHTML:#move-panel")

    # ── Rotation mode ──
    if GS.rotationClickId is not None:
        src_idx = GS.rotationClickId
        GS.rotationClickId = None
        src_hp = GS.parts.i2hp(src_idx)
        dst_hp = GS.parts.i2hp(hex_id)
        facing = next(
            (d for d in HexPosition.directions() if src_hp + d == dst_hp), None)
        if facing is not None:
            piece = next(
                (p for p in GS.player_squad.pieces
                 if p.place is not None
                 and GS.parts.hp2i(p.place.location) == src_idx), None)
            if piece:
                piece.place.facing = facing
                GS.pending_rotations = [r for r in GS.pending_rotations if r.piece is not piece]
                GS.pending_rotations.append(Rotation(piece, facing))
                log.info(f"  → rotated {piece.name} to face {facing.label}")
        return center(), left_oob()

    # ── Purchase placement ──
    if GS.pending_purchase_type is not None and GS.pending_purchase is None:
        loc = GS.parts.location_by_index(hex_id)
        king = next((p for p in GS.player_squad.pieces
                     if p.type == PieceType.KING and p.place is not None), None)
        facing = HexPosition.directions()[0]
        if king is not None:
            king_hp = king.place.location
            dst_hp = GS.parts.i2hp(hex_id)
            facing = min(HexPosition.directions(),
                         key=lambda d: (dst_hp + d).distance(king_hp))
        GS.pending_purchase = Purchase(GS.pending_purchase_type, loc, facing)
        GS.pending_purchase_type = None
        log.info(f"  → purchase placed @ hex {hex_id}")
        return center(), left_oob()

    # ── Click own piece → rotation mode ──
    clicked_piece = next(
        (p for p in GS.player_squad.pieces
         if p.place is not None and GS.parts.hp2i(p.place.location) == hex_id), None)
    if clicked_piece and GS.selected_piece is None and GS.pending_placement is None:
        GS.rotationClickId = hex_id
        log.info(f"  → rotation mode for {clicked_piece.name}")
        return center(), left_oob()

    # ── Placement / unplace ──
    if GS.selected_piece is not None:
        GS.place_piece(hex_id)
    elif GS.pending_placement is not None:
        placed_idx = GS.parts.hp2i(GS.pending_placement.piece.place.location)
        if placed_idx == hex_id:
            GS.unplace_piece(GS.pending_placement.piece)

    return center(), left_oob()


In [ ]:
@rt("/hex_click", methods=["POST"])
def hex_click(session, hex_id: int):
    gs, redir = get_gs_or_redirect(session)
    if redir: return redir
    pm = gs.pendingMove
    log.info(f"HEX_CLICK hex_id={hex_id} rotationMode={gs.rotationClickId}")

    def center():  
        return Div(panel_center(session), id="center-panel")
        
    def move_oob(): return Div(gs.move_panel(), hx_swap_oob="outerHTML:#move-panel")

    # ── Rotation mode ──
    if gs.rotationClickId is not None:
        src_idx = gs.rotationClickId
        gs.rotationClickId = None
        src_hp = gs.parts.i2hp(src_idx)
        dst_hp = gs.parts.i2hp(hex_id)
        facing = next(
            (d for d in HexPosition.directions() if src_hp + d == dst_hp), None)
        if facing is not None:
            piece = next(
                (p for p in gs.player_squad.pieces
                 if p.place is not None
                 and gs.parts.hp2i(p.place.location) == src_idx), None)
            if piece:
                piece.place.facing = facing
                pm.add_rotation(piece, facing)
                log.info(f"  → rotated {piece.name} to face {facing.label}")
        return center(), move_oob()

    # ── Purchase placement ──
    if pm.purchase_type is not None and pm.purchase is None:
        loc = gs.parts.location_by_index(hex_id)
        king = next((p for p in gs.player_squad.pieces
                     if p.type == PieceType.KING and p.place is not None), None)
        facing = HexPosition.directions()[0]
        if king is not None:
            king_hp = king.place.location
            dst_hp = gs.parts.i2hp(hex_id)
            facing = min(HexPosition.directions(),
                         key=lambda d: (dst_hp + d).distance(king_hp))
        pm.set_purchase(pm.purchase_type, loc, facing)
        log.info(f"  → purchase placed @ hex {hex_id}")
        return center(), move_oob()

    # ── Click own piece → rotation mode ──
    clicked_piece = next(
        (p for p in gs.player_squad.pieces
         if p.place is not None
         and gs.parts.hp2i(p.place.location) == hex_id), None)
    if clicked_piece and pm.selected_piece is None and pm.placement is None:
        gs.rotationClickId = hex_id
        log.info(f"  → rotation mode for {clicked_piece.name}")
        return center(), move_oob()

    # ── Placement ──
    
    if pm.selected_piece is not None:
        piece = pm.selected_piece  # grab ref before set_placement clears it
        loc = gs.parts.location_by_index(hex_id)
        piece.place = loc          # preview immediately
        pm.set_placement(piece, loc)

    elif pm.placement is not None:
        placed_idx = gs.parts.hp2i(pm.placement.piece.place.location)
        if placed_idx == hex_id:
            pm.placement.piece.place = None
            pm.placement = None
            log.info(f"  → unplaced piece")

    return center(), move_oob()


should hex_click be refactored?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The second version (using `pm`) is already much cleaner. But there's a **bug** in the placement section:

```python
if pm.selected_piece is not None:
    loc = gs.parts.location_by_index(hex_id)
    pm.set_placement(pm.selected_piece, loc)
    pm.selected_piece.place = loc  # ← BUG: selected_piece is None here!
```

`set_placement` sets `self.selected_piece = None`, so the next line crashes. Fix:

```python
    # ── Placement ──
    if pm.selected_piece is not None:
        piece = pm.selected_piece  # grab ref before set_placement clears it
        loc = gs.parts.location_by_index(hex_id)
        piece.place = loc          # preview immediately
        pm.set_placement(piece, loc)
```

One other thing — `rotationClickId` is UI-only transient state, same category as `selected_piece` and `purchase_type`. You could move it onto `PendingMove` and clear it in `pm.clear()` for consistency. But that's optional cleanup, not a bug.

## Move left side bar

@rt("/move/clear", methods=["POST"])
def move_clear(session):
    gs, redir = get_gs_or_redirect(session)
    if redir: return redir
    gs.pending_rotations = []
    gs.pending_placement = None
    gs.pending_purchase = None
    gs.pending_purchase_type = None
    gs.selected_piece = None
    return gs.move_panel()

@rt("/move/commit", methods=["POST"])
def move_commit(session):
    gs, redir = get_gs_or_redirect(session)
    if redir: return redir
    result = gs.commit_move()
    if result and 'error' in result:
        # Return the panel with an error toast, plus OOB map refresh
        error = P(f"⚠️ {result['error']}", cls="text-destructive text-sm font-medium")
        return gs_move_panel(gs), Div(error, id="move-error", hx_swap_oob="innerHTML:#move-error")
    # Refresh both the move panel and the map
    map_update = Div(panel_center(session), hx_swap_oob="outerHTML:#center-panel")
    return gs.move_panel(), map_update

@rt("/move/select_purchase", methods=["POST"])
def move_select_purchase(session, piece_type: str):
    gs, redir = get_gs_or_redirect(session)
    if redir: return redir
    gs.pending_purchase_type = PieceType(piece_type)
    return gs.move_panel()


In [ ]:
@rt("/move/clear", methods=["POST"])
def move_clear(session):
    gs, redir = get_gs_or_redirect(session)
    if redir: return redir
    gs.pendingMove.clear()
    return gs.move_panel()

@rt("/move/commit", methods=["POST"])
def move_commit(session):
    gs, redir = get_gs_or_redirect(session)
    if redir: return redir
    result = gs.commit_move()
    if result and 'error' in result:
        error_banner = P(f"⚠️ {result['error']}", cls="text-destructive text-sm font-medium")
        map_update = Div(gs.panel_map_content(), hx_swap_oob="outerHTML:#center-panel")
        return gs.move_panel(), map_update, error_banner
    map_update = Div(gs.panel_map_content(), hx_swap_oob="outerHTML:#center-panel")
    return gs.move_panel(), map_update

@rt("/move/select_purchase", methods=["POST"])
def move_select_purchase(session, piece_type: str):
    gs, redir = get_gs_or_redirect(session)
    if redir: return redir
    gs.pendingMove.purchase_type = PieceType(piece_type)
    return gs.move_panel()


refactor based on pending?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The routes are still using the old flat fields (`gs.pending_rotations`, `gs.pending_purchase_type`, etc.) instead of going through `gs.pendingMove`:

```python
@rt("/move/clear", methods=["POST"])
def move_clear(session):
    gs, redir = get_gs_or_redirect(session)
    if redir: return redir
    gs.pendingMove.clear()
    return gs.move_panel()

@rt("/move/commit", methods=["POST"])
def move_commit(session):
    gs, redir = get_gs_or_redirect(session)
    if redir: return redir
    result = gs.commit_move()
    if result and 'error' in result:
        error_banner = P(f"⚠️ {result['error']}", cls="text-destructive text-sm font-medium")
        map_update = Div(gs.panel_map_content(), hx_swap_oob="outerHTML:#center-panel")
        return gs.move_panel(), map_update, error_banner
    map_update = Div(gs.panel_map_content(), hx_swap_oob="outerHTML:#center-panel")
    return gs.move_panel(), map_update

@rt("/move/select_purchase", methods=["POST"])
def move_select_purchase(session, piece_type: str):
    gs, redir = get_gs_or_redirect(session)
    if redir: return redir
    gs.pendingMove.purchase_type = PieceType(piece_type)
    return gs.move_panel()
```

Key changes:
- **`clear`** → delegates to `PendingMove.clear()` which already resets all fields in one place
- **`select_purchase`** → sets `pendingMove.purchase_type` instead of the old flat field
- **`commit`** → always returns a map OOB update alongside the refreshed panel, whether there's an error or not — the map should reflect any previewed state either way
- **Removed the separate `#move-error` OOB target** — the error lives inside the move panel itself, which is simpler

## Main Window posts

In [ ]:
@rt("/toggle_debug", methods=["POST"])
def toggle_debug(session):
    GS, redir = get_gs_or_redirect(session)
    if redir: return redir
    GS.debug_overlays = not GS.debug_overlays
    log.info(f"DEBUG overlays={'ON' if GS.debug_overlays else 'OFF'}")
    return panel_center(session)


In [ ]:
@rt("/zoom", methods=["POST"])
def zoom(session,radius: int):
    GS, redir = get_gs_or_redirect(session)
    if redir: return redir
    GS.radius = max(10, min(45, radius))  # ← is this being called?
    log.info(f"ZOOM radius={GS.radius}")
    return panel_center(session)

In [ ]:
@rt("/toggle_overlay", methods=["POST"])
def toggle_overlay(session,overlay: str):
    GS, redir = get_gs_or_redirect(session)
    if redir: return redir
    log.info(f"TOGGLE overlay={overlay} active={GS.active_overlays}")
    if overlay in GS.active_overlays:
        GS.active_overlays.discard(overlay)
    else:
        GS.active_overlays.add(overlay)
    return panel_center(session)

In [ ]:
@rt("/piece/focus", methods=["POST"])
def piece_focus(session,piece_id: str):
    GS, redir = get_gs_or_redirect(session)
    if redir: return redir
    piece = next((p for p in GS.player_squad.pieces if p.id == piece_id), None)
    log.info(f"/piece/focus piece_id={piece_id}")
    if piece and piece.place is not None:
        GS.focused_piece = piece
    return panel_center(session,view="map")


In [ ]:
@rt("/piece/unfocus", methods=["POST"])
def piece_unfocus(session):
    GS, redir = get_gs_or_redirect(session)
    if redir: return redir
    GS.focused_piece = None
    return panel_center(session,view="map")


In [ ]:
@rt("/move/toggle_rotation", methods=["POST"])
def move_toggle_rotation(session, piece_id: str):
    gs, redir = get_gs_or_redirect(session)
    if redir: return redir
    pm = gs.pendingMove
    piece = next((p for p in pm.squad.pieces if p.id == piece_id), None)
    if piece is None: return gs.move_panel()

    existing = next((r for r in pm.rotations if r.piece is piece), None)
    if existing:
        pm.remove_rotation(piece)
    else:
        # Cycle clockwise by one step
        dirs = HexPosition.directions()
        cur_idx = dirs.index(piece.place.facing) if piece.place.facing in dirs else 0
        new_facing = dirs[(cur_idx + 1) % len(dirs)]
        pm.add_rotation(piece, new_facing)
        piece.place.facing = new_facing  # preview

    return gs.move_panel()


In [ ]:
@patch
def _map_controls(GS:GlobalGameState):
    """Zoom slider, overlay toggles, debug button."""
    gameplay_overlays = [
        ("terrain", "🏔️ Elevation"), 
        ("number", "🏔️ places"), ("food", "🌾 Food"),
        ("lanterns", "🔦 Sight"),     ("network", "🔗 Network"),
        ("optimal", "🎯 Optimal"),    ("names", "🏷️ Names"),
    ]
    detail_overlays = [
        ("ocean", "🌊 Ocean"),        ("soil", "🪨 Soil"),
        ("climate", "🌡️ Climate"),    ("temperature", "🌤️ Temp"),
        ("precipitation", "🌧️ Rain"),
    ]
    
    def overlay_row(items):
        return DivLAligned(
            *[Button(
                label,
                cls=(ButtonT.primary if key in GS.active_overlays else ButtonT.ghost, "text-xs"),
                hx_post="/toggle_overlay",
                hx_vals=json.dumps({"overlay": key}),
                hx_target="#center-panel",
            ) for key, label in items],
            cls="gap-1",
        )
    
    zoom_slider = Input(type="range", id="radius", name="radius",
        value=str(GS.radius), min="10", max="45", step="1",
        hx_post="/zoom", hx_trigger="change",
        hx_target="#center-panel", cls="uk-range flex-1")
    
    debug_button = Button(
        f"🐛 {'ON' if GS.debug_overlays else 'OFF'}",
        cls=(ButtonT.primary if GS.debug_overlays else ButtonT.ghost, "text-xs"),
        hx_post="/toggle_debug", hx_target="#center-panel",
    )
    
    return Div(
        DivLAligned(
            Small(f"🔍 {GS.radius}", cls=TextPresets.muted_sm),
            zoom_slider, debug_button, cls="flex-1 gap-2",
        ),
        overlay_row(gameplay_overlays),
        overlay_row(detail_overlays),
        cls="space-y-1 py-2",
    )

In [ ]:
CHART_COLS = [
    ('king_pantry',   '🍞 Pantry'),
    ('net_food',      '🌾 Net/turn'),
    ('total_harvest', '🌻 Harvest'),
    ('drain_taken',   '⚔️ Drain in'),
    ('drain_dealt',   '🗡️ Drain out'),
    ('pieces_placed', '♟ Pieces'),
    ('bottlenecks',   '🔗 Bottlenecks'),
    ('sight_coverage','👁️ Sight'),
    ('king_connected','🔗 Connected'),
    ('isolated',      '🏝️ Isolated'),
]

@rt("/toggle_chart_col", methods=["POST"])
def toggle_chart_col(session,col: str):
    GS, redir = get_gs_or_redirect(session)
    if redir: return redir

    if col in GS.charts.active_cols:
        if len(GS.charts.active_cols) > 1:  # always keep at least one
            GS.charts.active_cols.discard(col)
    else:
        GS.charts.active_cols.add(col)
    return GS.panel_charts()

@rt("/toggle_chart_squad", methods=["POST"])
def toggle_chart_squad(session,squad: str):
    GS, redir = get_gs_or_redirect(session)
    if redir: return redir

    if squad in GS.charts.active_squads:
        GS.charts.active_squads.discard(squad)
    else:
        GS.charts.active_squads.add(squad)
    return GS.panel_charts()


## Glue

So this need to be refactored a bunch
```python
@rt("/panel/center")
def panel_center(view: str = "map"):
    tabs = DivLAligned(
        Button("🗺️ Map",
               cls=(ButtonT.primary if view == "map" else ButtonT.ghost, "text-sm"),
               hx_get="/panel/center?view=map", hx_target="#center-panel",
               hx_swap="outerHTML"),
        Button("👥 Squad",
               cls=(ButtonT.primary if view == "squad" else ButtonT.ghost, "text-sm"),
               hx_get="/panel/center?view=squad", hx_target="#center-panel",
               hx_swap="outerHTML"),
        Button("🔗 Flow",
               cls=(ButtonT.primary if view == "flow" else ButtonT.ghost, "text-sm"),
               hx_get="/panel/center?view=flow", hx_target="#center-panel",
               hx_swap="outerHTML"),
        Button("📊 Charts",
               cls=(ButtonT.primary if view == "charts" else ButtonT.ghost, "text-sm"),
               hx_get="/panel/center?view=charts", hx_target="#center-panel",
               hx_swap="outerHTML"),
        cls="gap-1 pb-2",
    )
    
    if view == "squad":
        content = panel_squad_detail()
    elif view == "flow":
        content = panel_flow()
    elif view == "charts":
        content = panel_charts()
    else:
        content = panel_map_content()
    
    return Div(tabs, content, id="center-panel")
```
the charts panel should be from the MoveLog of the game for instance. I brought a bunch over and patched some into global game state. Let me know what you are missing. The left panel should just be the approraite MovePanel

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

A few things I can see are missing or broken before this can work cleanly:

**Undefined panel functions** — these are called but not yet shown:
- `panel_squad_detail()` 
- `panel_charts()` — you mentioned this should come from `MoveLog`; I'd need to see `MoveLog`'s API (specifically `moveLogDisplay()`, `purchaseTable()`, and what columns the underlying DataFrame has)
- `panel_pieces()` — referenced in `hex_click` OOB swaps

**Session threading gaps** — these routes reference bare `GS` or take `session` without declaring it as a parameter:
```python
# These all need `session` added as first param:
toggle_debug, zoom, toggle_overlay, piece_focus, piece_unfocus
```

**`_map_controls`** — captures `GS` from outer scope; needs to become `_map_controls(gs)` and be passed the looked-up session state.

**`panel_center` itself** needs `session` threaded through so it can pass `gs` down to each sub-panel and to `_map_controls`.

Could you share:
1. `??MoveLog` — specifically its DataFrame structure and any existing `__ft__` / display methods
2. What `panel_pieces` currently looks like (or should look like)?

That'll let me give you a complete refactored `panel_center` + all sub-panels in one go.

!cat ../../HexMagic/magic/moves.py

!cat ../../HexMagic/magic/turns.py

Does turn make more sense

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes, much more sense! `TurnProcessor` has richer data than `MoveLog` alone. Here's how the tabs map naturally:

- **"🗺️ Map"** → `panel_map_content(gs)`
- **"👥 Squad"** → `panel_squad_detail(gs)`
- **"📋 Log"** → `tp.turnLogDisplay(tp.summaries_by_turn)` — compact scrollable, newest first, chips + expandable detail
- **"📊 Table"** → `tp.turnTable(tp.summaries_by_turn)` — full turn × squad grid with `TurnCell` cards

And the refactored `panel_center` becomes clean:

```python
@rt("/panel/center")
def panel_center(session, view: str = "map"):
    gs, redir = get_gs_or_redirect(session)
    if redir: return redir
    tp = gs.tp

    def tab(label, v):
        active = view == v
        return Button(label,
            cls=(ButtonT.primary if active else ButtonT.ghost, "text-sm"),
            hx_get=f"/panel/center?view={v}",
            hx_target="#center-panel", hx_swap="outerHTML")

    tabs = DivLAligned(
        tab("🗺️ Map",    "map"),
        tab("👥 Squad",  "squad"),
        tab("📋 Log",    "log"),
        tab("📊 Table",  "table"),
        cls="gap-1 pb-2",
    )

    if view == "squad":
        content = panel_squad_detail(gs)
    elif view == "log":
        content = tp.turnLogDisplay(tp.summaries_by_turn)
    elif view == "table":
        content = tp.turnTable(tp.summaries_by_turn)
    else:
        content = panel_map_content(gs)

    return Div(tabs, content, id="center-panel")
```

`tp.record` (the `GameRecord`) is also there for sparklines/charts if you want a future **"📈 Stats"** tab — it has per-turn `king_pantry`, `net_food`, `drain_dealt` etc. across all squads ready for plotting.

The last missing piece is still `panel_squad_detail(gs)` — do you have that, or should I write it from scratch using `GameRecord.latest()`?

In [ ]:
@rt("/panel/center")
def panel_center(session, view: str = "map"):
    gs, redir = get_gs_or_redirect(session)
    if redir: return redir
    tp = gs.tp

    def tab(label, v):
        active = view == v
        return Button(label,
            cls=(ButtonT.primary if active else ButtonT.ghost, "text-sm"),
            hx_get=f"/panel/center?view={v}",
            hx_target="#center-panel", hx_swap="outerHTML")

    tabs = DivLAligned(
        tab("🗺️ Map",    "map"),
        tab("👥 Squad",  "squad"),
        tab("📋 Log",    "log"),
        tab("📊 Table",  "table"),
        cls="gap-1 pb-2",
    )

    if view == "squad":
        content = gs.panel_squad_detail()
    elif view == "log":
        content = tp.turnTable(tp.summaries_by_turn)
    elif view == "table":
        content = gs.panel_charts() 
    else:
        content = gs.panel_map_content()

    return Div(tabs, content, id="center-panel")


## Refactor Status — panel_center + left panel

### What's done
- `panel_center(session, view)` ✅ — tabs: Map / Squad / Log / Table
- `move_clear`, `move_commit`, `move_select_purchase` routes ✅
- `gs_move_panel(gs)` helper ✅
- `MoveLog`, `TurnProcessor`, `GameRecord`, `TurnCell` all reviewed ✅

### Still needed from codebase
- `panel_squad_detail(gs: GlobalGameState)` — body not yet seen
- `panel_map_content(gs)` — body not yet seen  
- `_map_controls(gs)` — needs `gs` param threaded in
- `GlobalGameState` dataclass — fields: `tp`, `parts`, `player_squad`,
  `pending_purchase_type`, `pending_purchase`, `pending_placement`,
  `pending_rotations`, `selected_piece`, `rotationClickId`,
  `radius`, `active_overlays`, `debug_overlays`, `focused_piece`
- `panel_pieces()` — referenced in `hex_click` OOB but not shown
- `get_gs_or_redirect(session)` — assumed to exist

### Key wiring decisions
- Left sidebar = `MovePanel(...)` with id `#move-panel`
- Log tab = `tp.turnLogDisplay(tp.summaries_by_turn)`
- Table tab = `tp.turnTable(tp.summaries_by_turn)`
- All routes need `session` as first param (not bare global `GS`)


Yes panel/history route should that show tp.turnLogDisplay(...)  can you write it?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

```python
@rt("/panel/history")
def panel_history(session):
    gs, redir = get_gs_or_redirect(session)
    if redir: return redir

    tp = gs.tp
    content = (
        tp.turnLogDisplay(tp.summaries_by_turn)
        if tp.summaries_by_turn
        else P("No turns played yet.", cls=TextPresets.muted_sm)
    )

    return Div(
        DivFullySpaced(
            H4("Turn History", cls="font-bold text-base"),
            Small(f"{gs.turn} turns", cls=TextPresets.muted_sm),
        ),
        Divider(cls="my-2"),
        content,
        id="history-panel",
        cls="space-y-2",
    )
```

After each `move_commit` you'll want to refresh it alongside the move panel and map. Add one more OOB swap to `move_commit`:

```python
@rt("/move/commit", methods=["POST"])
def move_commit(session):
    gs, redir = get_gs_or_redirect(session)
    if redir: return redir
    result = gs.commit_move()
    if result and 'error' in result:
        error = P(f"⚠️ {result['error']}", cls="text-destructive text-sm font-medium")
        return gs_move_panel(gs), Div(error, id="move-error", hx_swap_oob="innerHTML:#move-error")

    map_update     = Div(gs.panel_map_content(), hx_swap_oob="outerHTML:#center-panel")
    history_update = Div(panel_history(session),  hx_swap_oob="outerHTML:#history-panel")
    return gs_move_panel(gs), map_update, history_update
```

That way a single commit refreshes all three panels atomically.

In [ ]:
@rt
def game(session):
    gs, redir = get_gs_or_redirect(session)
    if redir: return redir

    return Title("HexMagic — Game"), Container(
        # ── Top bar ──
        DivFullySpaced(
            H3("⬡ HexMagic", cls="uk-h3"),
            Div(
                Span(f"Turn {gs.turn}", cls=TextT.lg),
                Span(f" · {gs.player_squad.name}", cls=TextPresets.muted_sm),
            ),
            Button("🏠 Menu", cls=ButtonT.ghost, hx_get="/", hx_target="body"),
            cls="py-2",
        ),
        Divider(),
        # ── Three-column layout ──
        Div(
            # Left: move panel (rendered immediately — no lazy load needed,
            # it's pure GS state with no heavy computation)
            Div(gs.move_panel(), id="move-panel",
                cls="w-72 flex-shrink-0 overflow-y-auto max-h-[80vh]"),

            # Center: map + tabs (lazy-loaded)
            Div(id="center-panel",
                hx_get="/panel/center", hx_trigger="load",
                hx_target="#center-panel", hx_swap="outerHTML",
                cls="flex-1 min-w-0"),

            # Right: turn history (lazy-loaded)
            Div(
                Div(id="history-panel",
                    hx_get="/panel/history", hx_trigger="load",
                    hx_target="#history-panel"),
                cls="w-[280px] flex-shrink-0 overflow-y-auto max-h-[80vh] space-y-3",
            ),
            cls="flex gap-3",
        ),
    )


In [ ]:
@rt("/victory")
def victory(session):
    gs, redir = get_gs_or_redirect(session)
    if redir: return redir
    return Title("Victory"), Container(
        DivCentered(
            H2("👑 Victory!", cls="text-yellow-500"),
            _game_over_summary(session),
            Button("🏠 New Game", cls=(ButtonT.primary, "mt-4"),
                   hx_get="/", hx_target="body"),
        )
    )

@rt("/eliminated")
def eliminated(session):
    gs, redir = get_gs_or_redirect(session)
    if redir: return redir
    return Title("Eliminated"), Container(
        DivCentered(
            H2("☠️ Your Kingdom Falls", cls="text-destructive"),
            P(f"The treasury ran dry on turn {gs.turn}.", cls=TextPresets.muted_lg),
            _game_over_summary(session),
            Button("🏠 New Game", cls=(ButtonT.primary, "mt-4"),
                   hx_get="/", hx_target="body"),
        )
    )


In [ ]:
@rt("/move/commit", methods=["POST"])
def move_commit(session):
    gs, redir = get_gs_or_redirect(session)
    if redir: return redir
    result = gs.commit_move()

    if result and 'error' in result:
        error = P(f"⚠️ {result['error']}", cls="text-destructive text-sm font-medium")
        return gs.move_panel(), Div(error, hx_swap_oob="innerHTML:#move-error")

    # Check game over
    if gs.tp.game_over:
        winner = gs.tp.winner
        if winner is gs.player_squad:
            return RedirectResponse("/victory", status_code=303)
        else:
            return RedirectResponse("/eliminated", status_code=303)
    
    center_update  = Div(panel_center(session),  hx_swap_oob="outerHTML:#center-panel")

    history_update = Div(panel_history(session),  hx_swap_oob="outerHTML:#history-panel")
    return gs.move_panel(), center_update, history_update


I don't think move_co

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

You'll just swap `gs_move_panel(gs)` for a direct `PurchasePanel` call:

```python
def gs_purchase_panel(gs: GlobalGameState) -> FT:
    sq = gs.player_squad
    king = next((p for p in sq.pieces if p.type == PieceType.KING and p.place), None)
    king_pantry = king.place.pantry if king else 0
    return Div(
        PurchasePanel(sq.flag, king_pantry, gs.pending_purchase_type),
        id="move-panel",
        cls="w-72 flex-shrink-0 overflow-y-auto max-h-[80vh]",
    )
```

Then in `game()`:

```python
Div(gs_purchase_panel(gs), id="move-panel",
    cls="w-72 flex-shrink-0 overflow-y-auto max-h-[80vh]"),
```

And update the three move routes to return `gs_purchase_panel(gs)` instead of `gs_move_panel(gs)`.

One thing to confirm — `PurchasePanel`'s buttons currently post to `/move/select_purchase` and the commit/clear are part of the full `MovePanel`. Do you want commit/clear buttons added below `PurchasePanel`, or will those live elsewhere (e.g. on the map itself or in the top bar)?

In [ ]:
@rt("/panel/history")
def panel_history(session):
    gs, redir = get_gs_or_redirect(session)
    if redir: return redir

    tp = gs.tp
    content = (
        tp.turnLogDisplay(tp.summaries_by_turn)
        if tp.summaries_by_turn
        else P("No turns played yet.", cls=TextPresets.muted_sm)
    )

    return Div(
        DivFullySpaced(
            H4("Turn History", cls="font-bold text-base"),
            Small(f"{gs.turn} turns", cls=TextPresets.muted_sm),
        ),
        Divider(cls="my-2"),
        content,
        id="history-panel",
        cls="space-y-2",
    )


!cat ../../HexMagic/magic/moves.py

I don't think def move_commit(session): is returning the correct middle panel

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The issue is `move_commit` is swapping in just `gs.panel_map_content()` — but `#center-panel` contains the **tabs + content** together. So the tabs disappear after a commit.

It should call the full `panel_center(session)` instead:

```python
@rt("/move/commit", methods=["POST"])
def move_commit(session):
    gs, redir = get_gs_or_redirect(session)
    if redir: return redir
    result = gs.commit_move()

    if result and 'error' in result:
        error = P(f"⚠️ {result['error']}", cls="text-destructive text-sm font-medium")
        return gs.move_panel(), Div(error, hx_swap_oob="innerHTML:#move-error")

    if gs.tp.game_over:
        winner = gs.tp.winner
        if winner is gs.player_squad:
            return RedirectResponse("/victory", status_code=303)
        else:
            return RedirectResponse("/eliminated", status_code=303)

    center_update  = Div(panel_center(session),  hx_swap_oob="outerHTML:#center-panel")
    history_update = Div(panel_history(session), hx_swap_oob="outerHTML:#history-panel")
    return gs.move_panel(), center_update, history_update
```

Same bug likely exists in `hex_click` — anywhere you have `gs.panel_map_content()` as an OOB swap target for `#center-panel`, it should be `panel_center(session)` instead.

In [ ]:
print('\n'.join(game_log))


In [ ]:
game_log=[]

## Debug

In [ ]:
#game_states

In [ ]:
debugGS = game_states['d47deff4-8839-47d3-b69f-6663442706f2']

In [ ]:
debugGS.player_squad.king_alive()

In [ ]:
# Who's still alive and where?
for p in debugGS.player_squad.pieces:
    alive = p.place is not None
    print(f"{p.type.icon} {p.name}: {'alive' if alive else 'DEAD'}"
          + (f" @ hex {debugGS.parts.hp2i(p.place.location)}, pantry={p.place.pantry}" if alive else ""))


Why does the king only have 2 pieces. Was he killed and came back to life?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Looking at `start_game`, each squad starts with **1 King + 3 Pawns** (from `place_optimal_pawns(k=3)`), so 4 pieces total. If the squad is now down to 2, those 2 pawns died.

The most likely cause is **food drain** during `play_round`. Each turn, enemy pieces in sight deal drain damage (`FoodStats.drain`), and if a piece's pantry hits 0, it starves. Pawns have `diet=1` per turn, so they're fragile if they're not connected to the king's food network.

You can check with:

```python
# Who's still alive and where?
for p in debugGS.player_squad.pieces:
    alive = p.place is not None
    print(f"{p.type.icon} {p.name}: {'alive' if alive else 'DEAD'}"
          + (f" @ hex {debugGS.parts.hp2i(p.place.location)}, pantry={p.place.pantry}" if alive else ""))
```

Also worth checking — looking at the game log, I see **tons** of "purchase placed @ hex" but no commits. Each `hex_click` that hits the purchase branch **overwrites** the previous pending purchase (`pm.set_purchase`). So if you were clicking around thinking you were buying multiple pieces, only the last one before commit would actually go through.

In [ ]:
show(debugGS.player_squad)

In [ ]:
debugGS.player_squad.table()

In [ ]:
debugGS.player_squad.animal

for piece in debugGS.player_squad.pieces:
    piece.animal =  debugGS.player_squad.animal

In [ ]:
??Piece